# Scoutra AI experiments on Kaggle 2×T4

Attach **zhanto-highlights**, select **GPU T4 x2**, turn on **Internet**, and choose **Run All**. The default `RUN_MODE="BENCHMARK"` checks prerequisites, runs a small RF-DETR Small smoke test on each physical GPU, and then automatically starts the full 12-clip benchmark. ByteTrack and BoT-SORT share the same frozen predictions. `RUN_MODE="SMOKE"` stops after the short test.

- **PREFLIGHT** checks CUDA in the isolated profile, verifies checkpoints, prepares dependencies, and audits videos when at least one method is ready.
- To run with Internet off, set `AUTO_ACQUIRE=False` and point `ASSET_CACHE` to an attached directory holding the verified `rf-detr-small.pth`. For a fresh session also point `WHEELHOUSE` to the dependency wheels. A valid profile and checkpoint already cached this session can be reused as is.
- **ADVANCED** runs selected optional methods. **REPORT** re-exports a saved run.
- If preflight or a requested method fails, the run saves its ZIP and reports **BLOCKED** or **FAILED**, and the execution cell raises an error. Open `RUN_RESULT.json`, `reports/failures.md` and `environment/network.json` in the ZIP.
- `benchmarks/device_frames.csv` shows the physical GPU UUID, worker PID and detector frames per device. GPU inference validation requires actual CUDA batches on every assigned device. The CPU validation in Codex Cloud does not establish Kaggle T4 performance.

For Python 3.13 setup, offline wheels, checkpoint URLs and the SHA-256, see the [launch and repair guide](https://github.com/erbo-dev/Scoutra/blob/research/scoutra-kaggle-runtime-repair/experiments/scoutra-dual-t4/README.md).


In [ ]:
# Run this cell after attaching zhanto-highlights and selecting Kaggle GPU T4 x2.
RUN_MODE = "BENCHMARK"               # Successful per-device smoke automatically starts all 12 clips.
AUTO_ACQUIRE = True                  # Enable Kaggle Internet, or set False with ASSET_CACHE and WHEELHOUSE.
USE_BOTH_GPUS = True
ALLOW_SINGLE_GPU = False             # Explicit consent to one-GPU fallback or a single-GPU run.
DATASET_ROOT = "auto"
MAX_VIDEOS = 12                      # Keep 12 for the full benchmark.
MODEL_GROUPS = ["detection", "tracking"]
DETECTOR_METHODS = ["rfdetr_small"]  # Add rfdetr_medium, dfine_small, rtdetr_r18vd, rtdetrv2_r18vd.
TRACKER_METHODS = ["bytetrack", "botsort_cmc"]  # strongsort needs OSNET_CHECKPOINT.
TRACK_DETECTOR = "rfdetr_small"      # The sole common detector cache for tracker comparison.
SCALING_MODELS = ["rfdetr_small"]    # BENCHMARK compares this exact workload on 2 GPUs and 1 GPU.
SAVE_ANNOTATED_VIDEOS = True

# Keep the first smoke small. Start at FP32; enable FP16 only with comparison tests.
SMOKE_FRAMES = 8
BATCH_SIZE = 4
PRECISION = "fp32"
DETECTOR_SCORE_FLOOR = 0.05
OPERATING_SCORE = 0.25
TRACKER_SETTINGS = {"high": 0.25, "low": 0.10, "new": 0.25, "match": 0.80, "buffer": 30}
JOB_TIMEOUT_S = 900
ASSET_CACHE = None                   # Read-only Kaggle folder containing rf-detr-small.pth.
WHEELHOUSE = None                    # Optional Kaggle folder with Linux CPython 3.13 dependency wheels.
CACHE_RUN = None                    # Prior run ID or extracted run directory for frozen detections/tracks.
REPORT_RUN = None                   # REPORT uses the latest saved run by default.

# Run advanced methods independently via RUN_MODE="ADVANCED" and this explicit list.
ADVANCED_METHODS = []               # Example ["no_link", "temporal_ball_baseline"] with CACHE_RUN set.
IDENTITY_TRACKER = "bytetrack"
IDENTITY_THRESHOLD = 0.90
IDENTITY_SWEEP = []                  # Optional additional cosine_tracklet_link thresholds, e.g. [0.88, 0.92, 0.95].
CROP_MANIFEST = None                # Or select prepare_crops before OSNet/part extraction in the same run.
CROP_STRIDE = 5
CROPS_PER_TRACK = 12
IDENTITY_FEATURES = None            # Saved, clip/track-fingerprinted global NPZ for identity linking.
LEGACY_REID = {}                    # See repository guide for exact PRTReID/BPBReID preparation.
DEEP_EIOU_CONFIG = {}               # Optional isolated tracker checkout and sports checkpoint.
GTA_CONFIG = {}                     # Optional isolated GTA-Link source/checkpoint configuration, see guide.
PITCH_LANDMARKS = None
OCR_PREDICTIONS = None
SCOUTRA_CHECKPOINT = None           # {'path','sha256','architecture','resolution','class_map','provenance'}
OSNET_CHECKPOINT = None             # {'path','sha256','provenance'}, require verified upstream OSNet filename.

DETECTION_ANNOTATIONS = None
TRACKING_ANNOTATIONS = None
REID_ANNOTATIONS = None
IDENTITY_ANNOTATIONS = None
REID_DISTANCE_THRESHOLD = None     # Set only from a held-out validation split before calculating FMR.
PART_VISIBILITY_MIN = 0.5
DISTANCE_SAMPLE_CROPS = 1200

CONFIG = {
    "mode": RUN_MODE, "auto_acquire": AUTO_ACQUIRE, "use_both": USE_BOTH_GPUS, "allow_single": ALLOW_SINGLE_GPU,
    "dataset_root": DATASET_ROOT, "max_videos": MAX_VIDEOS, "groups": MODEL_GROUPS,
    "detectors": DETECTOR_METHODS, "trackers": TRACKER_METHODS, "track_detector": TRACK_DETECTOR,
    "scaling_models": SCALING_MODELS, "save_videos": SAVE_ANNOTATED_VIDEOS, "smoke_frames": SMOKE_FRAMES,
    "batch": BATCH_SIZE, "precision": PRECISION, "score": DETECTOR_SCORE_FLOOR, "operating_score": OPERATING_SCORE,
    "tracker_settings": TRACKER_SETTINGS, "timeout": JOB_TIMEOUT_S, "asset_cache": ASSET_CACHE, "wheelhouse": WHEELHOUSE,
    "cache_run": CACHE_RUN, "report_run": REPORT_RUN, "advanced_methods": ADVANCED_METHODS,
    "identity_tracker": IDENTITY_TRACKER, "identity_threshold": IDENTITY_THRESHOLD, "identity_sweep": IDENTITY_SWEEP,
    "crop_manifest": CROP_MANIFEST, "crop_stride": CROP_STRIDE, "crops_per_track": CROPS_PER_TRACK,
    "identity_features": IDENTITY_FEATURES, "legacy_reid": LEGACY_REID, "gta_config": GTA_CONFIG, "deep_eiou_config": DEEP_EIOU_CONFIG,
    "landmarks": PITCH_LANDMARKS, "ocr_predictions": OCR_PREDICTIONS, "scoutra_checkpoint": SCOUTRA_CHECKPOINT,
    "osnet_checkpoint": OSNET_CHECKPOINT, "detection_annotations": DETECTION_ANNOTATIONS,
    "tracking_annotations": TRACKING_ANNOTATIONS, "reid_annotations": REID_ANNOTATIONS,
    "identity_annotations": IDENTITY_ANNOTATIONS, "reid_threshold": REID_DISTANCE_THRESHOLD,
    "visibility_min": PART_VISIBILITY_MIN, "distance_sample": DISTANCE_SAMPLE_CROPS,
}


## Embedded implementation

Run the following cells as part of **Run All**. They create plain Python files under the notebook's working directory so CUDA workers can start in independent Python processes. No second source Dataset or GitHub checkout is required.


In [ ]:
SOURCES = {}


In [ ]:
SOURCES['common.py'] = r'''"""Shared contracts. Importing this file never initializes CUDA."""
import csv
import http.client
import hashlib
import json
import math
import os
import re
import tempfile
import urllib.error
import urllib.request
import zipfile
from contextlib import contextmanager
from pathlib import Path

CLIP_SUFFIXES = (5479, 5480, 5483, 5485, 5487, 5491, 5495, 5497, 5503, 5504, 5505, 5513)
NAMES = tuple(f"document_544086323054304{s}.mp4" for s in CLIP_SUFFIXES)
BOX = ("x1", "y1", "x2", "y2")
STATES = {"EXECUTED", "FAILED", "SKIPPED", "BLOCKED", "NOT_EVALUATED"}
GROUPS = {"detection", "tracking", "reid", "identity", "jersey", "calibration", "ball", "events", "vlm"}


class Blocked(RuntimeError):
    """An explicit prerequisite is missing."""


def digest(path, algorithm="sha256"):
    with Path(path).open("rb") as handle:
        value = hashlib.new(algorithm)
        while chunk := handle.read(1024 * 1024):
            value.update(chunk)
        return value.hexdigest()


@contextmanager
def atomic_path(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, name = tempfile.mkstemp(prefix=f".{path.stem}-", suffix=path.suffix, dir=path.parent)
    os.close(fd)
    temporary = Path(name)
    try:
        yield temporary
        temporary.replace(path)
    finally:
        temporary.unlink(missing_ok=True)


def json_write(path, data):
    with atomic_path(path) as temporary:
        temporary.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def json_read(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def table(path, records, fields):
    with atomic_path(path) as temporary, temporary.open("w", newline="", encoding="utf-8") as handle:
        out = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
        out.writeheader()
        out.writerows(records)


def verified(path, sha256):
    path = Path(path)
    if not re.fullmatch(r"[a-f0-9]{64}", sha256 or ""):
        raise Blocked(f"Expected SHA-256 is required for {path.name}")
    if not path.is_file():
        raise Blocked(f"Required file missing: {path}")
    actual = digest(path)
    if actual != sha256:
        raise ValueError(f"SHA-256 mismatch for {path}: expected {sha256}, got {actual}")
    return path.resolve()


def fetch(url, path, sha256, online, mirrors=(), probe=None):
    path = Path(path)
    if path.is_file():
        try:
            return verified(path, sha256)
        except ValueError:
            if not online:
                raise
            # Replace a damaged writable cache only after verifying the new bytes.
    if not online:
        raise Blocked(f"Missing {path.name}. Set ASSET_CACHE to its verified local directory or enable Kaggle Internet and AUTO_ACQUIRE.")
    errors = []
    attempted = False
    for source in dict.fromkeys((url, *mirrors)):
        if probe:
            try:
                probe(source)
            except Blocked as error:
                errors.append(str(error))
                continue
        attempted = True
        for _ in range(2):
            try:
                with atomic_path(path) as temporary:
                    with urllib.request.urlopen(source, timeout=30) as response, temporary.open("wb") as out:
                        while chunk := response.read(1024 * 1024):
                            out.write(chunk)
                    verified(temporary, sha256)
                return path.resolve()
            except (OSError, http.client.HTTPException, ValueError) as error:
                errors.append(f"{source}: {error}")
    error_type = RuntimeError if attempted else Blocked
    raise error_type(f"Could not acquire verified {path.name}. {' | '.join(errors)}")


def probe_endpoint(url):
    request = urllib.request.Request(url, headers={"Range": "bytes=0-0", "User-Agent": "scoutra-kaggle-preflight/1"})
    with urllib.request.urlopen(request, timeout=8) as response:
        if not response.read(1):
            raise OSError("Empty response")
        return {"http_status": response.status, "reachable": True}


def resolve_dataset(request, mount=Path("/kaggle/input")):
    mount = Path(mount)
    if request != "auto":
        root = Path(request).resolve()
        missing = [name for name in NAMES if not (root / name).is_file()]
        if missing:
            raise Blocked(f"Explicit DATASET_ROOT {root} is missing: {', '.join(missing)}")
        return root
    direct = mount / "datasets/erbosynshaimardan/zhanto-highlights"
    if all((direct / name).is_file() for name in NAMES):
        return direct
    candidates = {path.parent.resolve() for path in mount.rglob(NAMES[0])}
    complete = sorted(path for path in candidates if all((path / name).is_file() for name in NAMES))
    if len(complete) == 1:
        return complete[0]
    if complete:
        raise Blocked(f"Multiple complete datasets: {complete}. Set DATASET_ROOT explicitly.")
    partial = {str(path): [name for name in NAMES if not (path / name).is_file()] for path in sorted(candidates)}
    raise Blocked(f"No directory with all 12 exact clips in {mount}. Attach zhanto-highlights. Partial candidates: {partial}")


def canonical_box(values, size):
    w, h = size
    x1, y1, x2, y2 = map(float, values)
    if not all(map(math.isfinite, (x1, y1, x2, y2))):
        raise ValueError("Nonfinite bounding box")
    x1, x2 = max(0, x1), min(w, x2)
    y1, y2 = max(0, y1), min(h, y2)
    if x2 <= x1 or y2 <= y1:
        return None
    return dict(zip(BOX, (x1, y1, x2, y2)))


def valid_score(score):
    score = float(score)
    if not 0 <= score <= 1:
        raise ValueError(f"Invalid confidence {score}")
    return score


def json_lines(path):
    with Path(path).open(encoding="utf-8") as handle:
        for line in handle:
            yield json.loads(line)


def write_line(handle, row):
    handle.write(json.dumps(row, separators=(",", ":"), allow_nan=False) + "\n")


def freeze_predictions(path, video):
    """Read consecutive, validated frames and verify physical clip provenance."""
    path = Path(path)
    meta = json_read(path.with_suffix(".json"))
    if meta["clip_sha256"] != video["sha256"] or meta["clip"] != video["clip"]:
        raise ValueError("Frozen detection cache belongs to another clip")
    if digest(path) != meta["prediction_sha256"]:
        raise ValueError("Frozen detection cache failed SHA-256 check")
    count = 0
    for count, row in enumerate(json_lines(path), 1):
        if row["frame"] != count:
            raise ValueError("Detection cache has missing or nonconsecutive frames")
        for index, box in enumerate(row["detections"]):
            if box["detection_id"] != f"{count}:{index}":
                raise ValueError("Detection provenance ID disagrees with row ordering")
            if box["class"] not in {"person", "player", "goalkeeper", "referee", "ball"}:
                raise ValueError("Unexpected canonical class in detection cache")
            valid_score(box["score"])
            coordinates = {key: box[key] for key in BOX}
            raw_id = box.get("raw_id")
            if canonical_box(coordinates.values(), (video["width"], video["height"])) != coordinates:
                raise ValueError("Invalid native-pixel box in frozen predictions")
            if type(raw_id) is not int:
                raise ValueError("Invalid raw class ID in frozen predictions")
        yield row
    if count != meta["frames"] or count < 1:
        raise ValueError("Detection cache is empty or row count disagrees with metadata")


def export_zip(run, output):
    run, output = Path(run), Path(output)
    with atomic_path(output) as temporary, zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED, compresslevel=3) as archive:
        for path in sorted(run.rglob("*")):
            relative = path.relative_to(run)
            if path.is_file() and not any(part.startswith(".") or part == "export" for part in relative.parts):
                archive.write(path, relative.as_posix())
    return output
'''


In [ ]:
SOURCES['backends.py'] = r'''"""Pinned model and tracker interfaces. COCO and football-trained labels stay explicit."""
from pathlib import Path
from types import SimpleNamespace

from common import BOX, Blocked, canonical_box, fetch, valid_score, verified

RF_COMMIT = "f29e2a8bbb7cdcb8920bf74a335062f88a35046d"
HF_COMMIT = "753d61104116eefc8ffc977327b441ee0c8d599f"
ULTRA_COMMIT = "4b884f4a44685f6ea45f4676c1bca50a2d72d443"
RF_TRANSFORMERS_COMMIT = "ddb849abe009d1089e6c691bfc897f27211c663c"
OSNET_SHA256 = "6f57607fed9f502b9efed546108132ee715df5a5b6e6932c6269bacb47f59f99"
BOXMOT_COMMIT = "c877349157a35df6095e703e002c60418d2b34ab"
RF = {
    "rfdetr_small": ("small", "d81979a9213a2109345158ce9232668df4c1ae52e9b8db3f2ec0a8cbad959b33"),
    "rfdetr_medium": ("medium", "749ff6071828aaffac63e204c4f4135ed3d6cdae4d702e086c360edc3b5768c8"),
}
HF = {
    "dfine_small": (
        "ustc-community/dfine-small-coco", "f79e65b5fbb33ceb9d3ebba042955d7410c608f8", "d_fine",
        "9f206c4c935c67c957af2f15c3333b4ff224e72b1881bc7bbde9f873edf4448e",
        "7e9e2245806aa3ef068a5321f5eebee3116fa4cec560173231954ffaca930eca",
        "a144baff3a27bfd538fe37b11fa8d4fca5744b8c054b5301e7c699a1379cd18f",
    ),
    "rtdetr_r18vd": (
        "PekingU/rtdetr_r18vd", "ac77a11ff0170a41b771c03264987f8ce2b0d753", "rt_detr",
        "8493be71f51a1c0a741f8f71ec151039227579379de7bcba047c0470d9320c3c",
        "ffb4b9461a1dad746be8f0f9c8330ed7743a1ba5fba4f75c232cd281b3d4c64a",
        "fe87a5a30f5daf298d10794c7682a63b6107986f97d6a770ba948d89e4340093",
    ),
    "rtdetrv2_r18vd": (
        "PekingU/rtdetr_v2_r18vd", "5650961749fa93567c0d46fc7f43ea4f9e914107", "rt_detr_v2",
        "ed051ec77cb41c5d9d5e3af21a979b1e890599dfd68434d7636ff82ded4c1527",
        "cd38cd59999e7a95d68e487fbe5132df3d4e5c32a0836add57e6126ba0c4eaf1",
        "d18309d0d7ea57048138885c4c6ecfcb1e24506fc6153b94ad484f8ab62c7115",
    ),
}
DETECTORS = {**dict.fromkeys(RF, "rf"), **dict.fromkeys(HF, "hf"), "scoutra_rfdetr": "rf"}
TRACKERS = {"bytetrack": "tracker", "botsort_cmc": "tracker", "strongsort": "boxmot"}
CLASS_MAP = {"person": "person", "sports ball": "ball"}
HUMANS = {"person", "player", "goalkeeper", "referee"}


def detector_asset(name, config, cache, probe=None):
    offline = Path(config["asset_cache"]) if config.get("asset_cache") else None
    online = config["auto_acquire"]
    if name in RF:
        size, checksum = RF[name]
        release = f"{size}_coco/checkpoint_best_regular.pth"
        url = f"https://repo.roboflow.com/rfdetr/{release}"
        mirror = f"https://storage.googleapis.com/rfdetr/{release}"
        filename = f"rf-detr-{size}.pth"
        local = offline / filename if offline else None
        if local and local.is_file():
            path = verified(local, checksum)
        else:
            path = fetch(url, cache / filename, checksum, online, mirrors=(mirror,), probe=probe)
        return dict(path=str(path), weights_sha256=checksum, weights_source=url, weights_mirrors=[mirror], source_commit=RF_COMMIT,
                    checkpoint_kind="COCO", checkpoint_license="Apache-2.0",
                    transformers_commit=RF_TRANSFORMERS_COMMIT)
    if name in HF:
        repo, revision, model_type, *hashes = HF[name]
        url = f"https://huggingface.co/{repo}/resolve/{revision}"
        files = dict(zip(("config.json", "preprocessor_config.json", "model.safetensors"), hashes))
        local = offline / name if offline else None
        if local and all((local / filename).is_file() for filename in files):
            for filename, checksum in files.items():
                verified(local / filename, checksum)
            directory = local.resolve()
        else:
            directory = cache / name
            for filename, checksum in files.items():
                fetch(f"{url}/{filename}", directory / filename, checksum, online, probe=probe)
        return dict(path=str(directory.resolve()), files=files, weights_source=repo, weights_revision=revision,
                    weights_sha256=hashes[-1], source_commit=HF_COMMIT, model_type=model_type,
                    checkpoint_kind="COCO", checkpoint_license="Apache-2.0")
    spec = config["scoutra_checkpoint"]
    if not spec:
        raise Blocked("Set SCOUTRA_CHECKPOINT with trusted path, SHA-256, class map, base architecture and resolution.")
    required = {"path", "sha256", "architecture", "resolution", "class_map", "provenance"}
    if not required <= set(spec) or spec["architecture"] not in {"RFDETRSmall", "RFDETRMedium", "RFDETRLarge", "RFDETRBase", "RFDETRNano"}:
        raise Blocked(f"SCOUTRA_CHECKPOINT requires {sorted(required)} and a supported RF-DETR architecture.")
    if not spec["class_map"] or not set(spec["class_map"].values()) <= HUMAN_AND_BALL:
        raise Blocked("Custom class_map must match training IDs and use canonical labels person/player/goalkeeper/referee/ball.")
    path = verified(spec["path"], spec["sha256"])
    return dict(path=str(path), weights_sha256=spec["sha256"], weights_source=spec["provenance"],
                source_commit=RF_COMMIT, checkpoint_kind="Scoutra fine-tuned (user-supplied)",
                architecture=spec["architecture"], resolution=spec["resolution"], class_map=spec["class_map"],
                trust_checkpoint=bool(spec.get("trust_checkpoint", False)))


HUMAN_AND_BALL = HUMANS | {"ball"}


def osnet_asset(config):
    spec = config["osnet_checkpoint"]
    if not spec or not spec.get("provenance"):
        raise Blocked("Provide a rights-reviewed OSNET_CHECKPOINT with path, SHA-256 and provenance.")
    if spec.get("sha256") != OSNET_SHA256:
        raise Blocked(f"The isolated BoxMOT condition accepts the verified upstream OSNet checkpoint SHA-256 {OSNET_SHA256}.")
    path = verified(spec["path"], OSNET_SHA256)
    if path.name != "osnet_x0_25_msmt17.pt":
        raise Blocked("BoxMOT infers architecture from filename. Keep the audited osnet_x0_25_msmt17.pt filename.")
    return dict(path=str(path), weights_sha256=spec["sha256"], weights_source=spec["provenance"],
                source_commit=BOXMOT_COMMIT, checkpoint_kind="OSNet x0.25 MSMT17 (user supplied)")


class Detector:
    def __init__(self, name, asset, device, precision, score):
        import torch

        self.name, self.device = name, torch.device(device)
        self.precision, self.score = precision, score
        self.load_info = {}
        self.is_rf = name not in HF
        if self.is_rf:
            import rfdetr

            if name == "scoutra_rfdetr":
                self.model = rfdetr.RFDETR.from_checkpoint(
                    asset["path"], device=device, trust_checkpoint=asset["trust_checkpoint"]
                )
                if type(self.model).__name__ != asset["architecture"] or self.model.model_config.resolution != asset["resolution"]:
                    raise ValueError("Scoutra checkpoint architecture or input resolution disagrees with the declared settings")
                self.custom = {int(k): value for k, value in asset["class_map"].items()}
                labels = self.model.class_names
                if len(labels) != len(self.custom) or set(self.custom) != set(range(len(labels))):
                    raise ValueError("Scoutra class map must cover all contiguous checkpoint classes")
                self.load_info["checkpoint_class_names"] = labels
            else:
                cls = {"rfdetr_small": rfdetr.RFDETRSmall, "rfdetr_medium": rfdetr.RFDETRMedium}[name]
                self.model = cls(pretrain_weights=asset["path"], device=device)
                self.custom = None
                from rfdetr.assets.coco_classes import COCO_CLASSES
                if COCO_CLASSES.get(1) != "person" or COCO_CLASSES.get(37) != "sports ball":
                    raise ValueError("RF-DETR sparse COCO map changed")
                self.load_info["raw_id_map"] = {1: "person", 37: "ball"}
            if next(self.model.model.model.parameters()).device != self.device:
                raise RuntimeError("RF-DETR parameters are not on their assigned device")
            self.load_info["input_resolution"] = self.model.model_config.resolution
        else:
            from transformers import (DFineForObjectDetection, RTDetrForObjectDetection,
                                      RTDetrImageProcessor, RTDetrV2ForObjectDetection)

            cls = {"d_fine": DFineForObjectDetection, "rt_detr": RTDetrForObjectDetection,
                   "rt_detr_v2": RTDetrV2ForObjectDetection}[asset["model_type"]]
            self.processor = RTDetrImageProcessor.from_pretrained(asset["path"], local_files_only=True)
            self.model, info = cls.from_pretrained(
                asset["path"], local_files_only=True, use_safetensors=True, output_loading_info=True
            )
            for key in ("missing_keys", "mismatched_keys", "error_msgs"):
                if info.get(key):
                    raise ValueError(f"{name}: incompatible checkpoint: {key}={info[key]}")
            self.labels = {int(k): value for k, value in self.model.config.id2label.items()}
            if self.model.config.model_type != asset["model_type"] or {k: self.labels.get(k) for k in (0, 32)} != {0: "person", 32: "sports ball"}:
                raise ValueError(f"{name}: incorrect model architecture or COCO label map")
            self.model = self.model.to(self.device).eval()
            self.load_info = {"unexpected_keys": info.get("unexpected_keys", []), "raw_id_map": {0: "person", 32: "ball"},
                              "input_resolution": self.processor.size}

    def predict(self, rgb):
        import torch

        with torch.inference_mode(), torch.autocast(self.device.type, dtype=torch.float16, enabled=self.precision == "fp16"):
            if self.is_rf:
                outputs = self.model.predict(rgb, threshold=self.score, include_source_image=False)
                raw = []
                for out in outputs:
                    labels = out.data.get("class_name")
                    if len(out) and labels is None:
                        raise ValueError("RF-DETR did not export class names")
                    raw.append(zip(out.xyxy, out.confidence, out.class_id, labels if labels is not None else []))
            else:
                inputs = self.processor(images=rgb, return_tensors="pt")
                if self.device.type == "cuda":
                    inputs = {key: tensor.pin_memory().to(self.device, non_blocking=True) for key, tensor in inputs.items()}
                outputs = self.model(**inputs)
                sizes = torch.as_tensor([im.shape[:2] for im in rgb], device=self.device)
                predictions = self.processor.post_process_object_detection(outputs, threshold=self.score, target_sizes=sizes)
                raw = []
                for out in predictions:
                    ids = out["labels"].detach().cpu().tolist()
                    raw.append(zip(out["boxes"].detach().cpu().numpy(), out["scores"].detach().cpu().numpy(),
                                   ids, (self.labels[i] for i in ids)))
        if len(raw) != len(rgb):
            raise ValueError("Model did not return one prediction object per image")
        return [self._normalize(rows, (image.shape[1], image.shape[0])) for image, rows in zip(rgb, raw)]

    def _normalize(self, rows, shape):
        result = []
        for values, score, raw_id, label in rows:
            raw_id = int(raw_id)
            canon = self.custom.get(raw_id) if self.name == "scoutra_rfdetr" else CLASS_MAP.get(str(label))
            if canon is None:
                continue
            if self.name in RF and {1: "person", 37: "ball"}.get(raw_id) != canon:
                raise ValueError(f"Unexpected RF-DETR class mapping {raw_id} -> {label}")
            box = canonical_box(values, shape)
            if box:
                result.append(dict(box, score=valid_score(score), raw_id=raw_id, raw_label=str(label), **{"class": canon}))
        return result


def format_tracker_parameters(name, thresholds, fps):
    return dict(track_high_thresh=thresholds["high"], track_low_thresh=thresholds["low"],
                new_track_thresh=thresholds["new"], match_thresh=thresholds["match"],
                track_buffer=thresholds["buffer"], fuse_score=True, with_reid=False,
                gmc_method="sparseOptFlow", proximity_thresh=0.5, appearance_thresh=0.8,
                model="auto", frame_rate=max(1, round(fps)))


class Tracker:
    def __init__(self, name, fps, settings, asset=None, device=None):
        self.name = name
        self.thresholds = settings
        parameters = format_tracker_parameters(name, settings, fps)
        if name == "strongsort":
            import torch
            from boxmot import StrongSort
            from advanced import verify_osnet_backbone

            self.model = StrongSort(reid_weights=Path(asset["path"]), device=torch.device(device), half=False)
            verify_osnet_backbone(self.model.model, Path(asset["path"]), device)
            self.parameters = {"implementation": "BoxMOT StrongSort upstream defaults", "source_commit": BOXMOT_COMMIT}
        else:
            from ultralytics.trackers.bot_sort import BOTSORT
            from ultralytics.trackers.byte_tracker import BYTETracker

            cls = {"bytetrack": BYTETracker, "botsort_cmc": BOTSORT}[name]
            fps = parameters.pop("frame_rate")
            self.model = cls(SimpleNamespace(**parameters), frame_rate=fps)
            self.parameters = {**parameters, "frame_rate": fps, "source_commit": ULTRA_COMMIT}

    def update(self, detections, frame):
        import numpy as np

        people = [d for d in detections if d["class"] in HUMANS]
        xyxy = np.array([[d[key] for key in BOX] for d in people], dtype=np.float32).reshape(-1, 4)
        confidence = np.array([d["score"] for d in people], dtype=np.float32)
        labels = np.zeros(len(people), dtype=np.float32)
        if self.name == "strongsort":
            out = self.model.update(np.column_stack((xyxy, confidence, labels)), frame)
        else:
            from ultralytics.engine.results import Boxes

            boxes = Boxes(np.column_stack((xyxy, confidence, labels)), orig_shape=frame.shape[:2])
            high = confidence >= self.thresholds["high"]
            high_indices = np.flatnonzero(high)
            low_indices = np.flatnonzero((confidence > self.thresholds["low"]) & ~high)
            out = self.model.update(boxes, frame)
        result = []
        for row in out:
            if len(row) != 8 or not np.isfinite(row).all():
                raise ValueError("Tracker must return Nx8 finite rows including the detection index")
            index = int(row[7])
            if self.name == "strongsort":
                ref = index
            else:
                # The pinned port assigns indices separately to its high- and low-score arrays.
                pool = high_indices if row[5] >= np.float32(self.thresholds["high"]) else low_indices
                if not 0 <= index < len(pool):
                    raise ValueError(f"Tracker index {index} is out of range in its score partition")
                ref = int(pool[index])
            if not 0 <= ref < len(people) or not np.isclose(row[5], confidence[ref], atol=1e-7, rtol=0):
                raise ValueError(f"Tracker provenance disagrees with the frozen input detection at index {ref}")
            box = canonical_box(row[:4], (frame.shape[1], frame.shape[0]))
            if box is not None:
                result.append(dict(box, track_id=int(row[4]), score=valid_score(row[5]),
                                   detection_id=people[ref]["detection_id"], **{"class": people[ref]["class"]}))
        if len({r["track_id"] for r in result}) != len(result):
            raise ValueError("Tracker emitted duplicate IDs within one frame")
        return result
'''


In [ ]:
SOURCES['environment.py'] = r'''"""Isolate packages per experiment and retain Kaggle's matched Torch/CUDA wheels."""
import ctypes
import importlib.metadata as metadata
import json
import os
import platform
import re
import subprocess
import sys
import hashlib
import uuid
import venv
from pathlib import Path

from common import Blocked, json_write

# Explicit scientific and upstream API pins. Freeze captures the full resolved graph.
BASE = ("numpy==2.2.6", "opencv-python==4.11.0.86", "pillow==11.3.0")
PACKAGE_PROFILES = {
    "rf": BASE + ("rfdetr==1.12.1", "transformers==5.12.1", "supervision==0.29.0"),
    "hf": BASE + ("transformers==4.57.6", "huggingface-hub==0.36.2", "safetensors==0.6.2"),
    "tracker": BASE + ("ultralytics==8.3.235", "lap==0.5.13"),
    "boxmot": BASE + ("boxmot==16.0.10", "lapx==0.5.12", "gdown==5.2.0"),
    "eval": ("numpy==2.3.5", "opencv-python==4.11.0.86", "pycocotools==2.0.10", "trackeval==1.3.0"),
}
IMPORT_SMOKES = {
    "rf": "from rfdetr import RFDETRSmall, RFDETRMedium, RFDETR; import cv2",
    "hf": "from transformers import DFineForObjectDetection, RTDetrForObjectDetection, RTDetrV2ForObjectDetection, RTDetrImageProcessor; import cv2",
    "tracker": "import lap; from ultralytics.trackers.byte_tracker import BYTETracker; from ultralytics.trackers.bot_sort import BOTSORT",
    "boxmot": "from boxmot import StrongSort; from boxmot.reid.core.auto_backend import ReidAutoBackend; import cv2",
    "eval": "from pycocotools.cocoeval import COCOeval; import trackeval",
}
NO_NET_ENV = {
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "YOLO_AUTOINSTALL": "false",
    "YOLO_OFFLINE": "true", "HF_HUB_DISABLE_TELEMETRY": "1", "OMP_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "TOKENIZERS_PARALLELISM": "false", "MPLBACKEND": "Agg",
    "PYTHONDONTWRITEBYTECODE": "1", "PYTHONNOUSERSITE": "1",
}


def command(argv, log, timeout=1200, environment=None):
    with Path(log).open("a", encoding="utf-8") as handle:
        handle.write(f"\n$ {argv!r}\n")
        handle.flush()
        result = subprocess.run(argv, stdout=handle, stderr=subprocess.STDOUT, timeout=timeout, env=environment)
        handle.write(f"\nexit={result.returncode}\n")
    return result.returncode


def cuda_uuid(torch, index):
    """Return the physical UUID in the process's visible-device order."""
    value = str(getattr(torch.cuda.get_device_properties(index), "uuid", ""))
    match = re.search(r"[0-9a-f]{8}(?:-[0-9a-f]{4}){3}-[0-9a-f]{12}", value, re.IGNORECASE)
    if match:
        return str(uuid.UUID(match.group()))
    class DeviceUUID(ctypes.Structure):
        _fields_ = [("data", ctypes.c_ubyte * 16)]
    driver = ctypes.CDLL("libcuda.so.1")
    dev, result = ctypes.c_int(), DeviceUUID()
    if driver.cuInit(0) or driver.cuDeviceGet(ctypes.byref(dev), index):
        raise RuntimeError(f"CUDA driver cannot resolve visible device {index}")
    get_uuid = getattr(driver, "cuDeviceGetUuid_v2", None) or driver.cuDeviceGetUuid
    if get_uuid(ctypes.byref(result), dev):
        raise RuntimeError(f"CUDA driver cannot read physical UUID for visible device {index}")
    return str(uuid.UUID(bytes=bytes(result.data)))


def inventory(path, python=sys.executable, env=None):
    """Run in a disposable subprocess so the notebook process has no CUDA context."""
    probe = """
import json, sys, traceback
from environment import cuda_uuid
result = {'python':sys.version, 'cuda_available':False, 'gpu_count':0, 'gpus':[]}
try:
 import torch, torchvision
 result.update(torch=torch.__version__, torchvision=torchvision.__version__, cuda_version=torch.version.cuda,
               cuda_available=torch.cuda.is_available(), gpu_count=torch.cuda.device_count())
 for i in range(result['gpu_count']):
  entry={'index':i}
  try:
   with torch.cuda.device(i):
    p=torch.cuda.get_device_properties(i)
    free,total=torch.cuda.mem_get_info(i)
    entry.update(name=p.name, uuid=cuda_uuid(torch,i),
                 total_bytes=p.total_memory, free_bytes_before_probe=free,
                 capability=[p.major,p.minor])
    x=torch.ones(16, device=f'cuda:{i}')
    assert (x+x).sum().item()==32
    torch.cuda.synchronize(i)
    entry['functional']=True
  except Exception:
   entry.update(functional=False,error=traceback.format_exc())
  result['gpus'].append(entry)
except Exception:
 result['error']=traceback.format_exc()
print(json.dumps(result))
"""
    result = subprocess.run([python, "-c", probe], capture_output=True, text=True, timeout=90,
                            cwd=Path(__file__).parent, env=env)
    if result.returncode or not result.stdout.strip():
        values = {"cuda_available": False, "gpu_count": 0, "gpus": [], "error": result.stderr}
    else:
        values = json.loads(result.stdout.strip().splitlines()[-1])
        if result.stderr:
            values["probe_stderr"] = result.stderr
    values.update(platform=platform.platform(), cuda_visible_devices=os.getenv("CUDA_VISIBLE_DEVICES"))
    try:
        query = subprocess.run(["nvidia-smi", "--query-gpu=index,uuid,name,driver_version,memory.total,memory.free",
                                "--format=csv,noheader,nounits"], text=True, capture_output=True, timeout=10)
        values["nvidia_smi"] = {"exit": query.returncode, "stdout": query.stdout, "stderr": query.stderr}
    except (OSError, subprocess.TimeoutExpired) as error:
        values["nvidia_smi"] = {"error": str(error)}
    json_write(path, values)
    return values


def device_plan(info, dual, allow_single):
    good = [f"cuda:{p['index']}" for p in info["gpus"] if p["functional"]]
    if dual and len(good) >= 2:
        return good[:2]
    if (allow_single or not dual) and good:
        return good[:1]
    raise Blocked(f"Need two functioning CUDA GPUs for dual mode. Found {good}. Select GPU T4 x2, or explicitly set ALLOW_SINGLE_GPU=True for one GPU.")


def _share_torch(site):
    """Expose only native runtime libraries, avoiding unrelated packages in Kaggle."""
    names = {"torch", "torchvision", "triton", "pytorch-triton", "pytorch-triton-rocm", "cuda-bindings", "cuda-pathfinder"}
    packages = {"torch", "torchgen", "functorch", "torchvision", "torchvision.libs", "triton", "nvidia", "cuda"}
    linked = set()
    for dist in metadata.distributions():
        name = dist.metadata["Name"].lower().replace("_", "-")
        if name not in names and not name.startswith("nvidia-"):
            continue
        for item in dist.files or []:
            top = item.parts[0]
            if top not in packages and not top.endswith(".dist-info"):
                continue
            source = Path(dist.locate_file(top))
            if source.is_dir() and top not in linked:
                target = site / top
                if not target.exists():
                    target.symlink_to(source.resolve(), target_is_directory=True)
                linked.add(top)
    if not {"torch", "torchvision"} <= linked:
        raise Blocked("Cannot share Kaggle's Torch/TorchVision runtime. Check the native CUDA probe and standard wheel installation.")
    return {name: metadata.version(name) for name in ("torch", "torchvision")}


def profile_support(profile, python_version):
    upper = (3, 12) if profile == "boxmot" else (3, 13)
    if not (3, 11) <= python_version[:2] <= upper:
        explanation = " BoxMOT 16.0.10 declares Python <3.13. The default RF-DETR and tracking paths use separate profiles." if profile == "boxmot" else ""
        wanted = ".".join(map(str, upper))
        actual = ".".join(map(str, python_version[:3]))
        raise Blocked(f"Profile {profile} requires Python 3.11–{wanted}, found Python {actual}.{explanation}")


def pip_environment():
    return {**{k: v for k, v in os.environ.items() if not k.startswith("PIP_")},
            "PIP_CONFIG_FILE": os.devnull, "PIP_DISABLE_PIP_VERSION_CHECK": "1",
            "PIP_NO_INPUT": "1", "PIP_TIMEOUT": "15", "PIP_RETRIES": "2"}


def prepare_profile(profile, workspace, run, online, wheelhouse=None, probe=None):
    profile_support(profile, sys.version_info)
    versions = {} if profile == "eval" else {name: metadata.version(name) for name in ("torch", "torchvision")}
    profile_digest = hashlib.sha256(
        repr(("host-pip-v1", PACKAGE_PROFILES[profile], versions, sys.version)).encode()
    ).hexdigest()[:12]
    root = workspace / "profiles" / f"{profile}-{profile_digest}"
    python = root / "bin/python"
    ready = root / ".validated"
    log = run / "environment" / "installation_log.txt"
    pip = [sys.executable, "-m", "pip", "--python", str(python)]
    environment = pip_environment()
    if not ready.is_file() or not python.is_file():
        ready.unlink(missing_ok=True)
        if wheelhouse:
            wheelhouse = Path(wheelhouse).resolve()
            if not wheelhouse.is_dir() or not any(wheelhouse.glob("*.whl")):
                raise Blocked(f"WHEELHOUSE must name a directory of compatible .whl files: {wheelhouse}")
            index = ["--no-index", "--find-links", str(wheelhouse)]
        else:
            if not online:
                raise Blocked(f"Profile {profile} is missing. Set WHEELHOUSE for a cold offline install, or enable Kaggle Internet and AUTO_ACQUIRE.")
            if probe:
                probe("https://pypi.org/simple/pip/")
            index = ["--index-url", "https://pypi.org/simple"]
        if command([sys.executable, "-m", "pip", "--version"], log, timeout=30, environment=environment):
            raise Blocked("The Kaggle Python interpreter needs pip with --python support to bootstrap a worker environment.")
        venv.EnvBuilder(with_pip=False).create(root)
        if versions:
            site = root / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
            native = _share_torch(site)
            constraints = root / "native-constraints.txt"
            constraints.write_text("".join(f"{name}=={version}\n" for name, version in native.items()))
            options = ["-c", str(constraints), "torch", "torchvision"]
        else:
            options = []
        binary = ["--only-binary=:all:"] if profile != "boxmot" else []
        if command([*pip, "install", *index, *binary, *options, *PACKAGE_PROFILES[profile]], log, environment=environment):
            raise RuntimeError(f"Dependency installation failed for {profile}; see {log} for exact resolver output.")
    yolo_dir = workspace / "ultralytics"
    if profile == "tracker":
        yolo_dir.mkdir(parents=True, exist_ok=True)
    env = {**os.environ, **NO_NET_ENV, "YOLO_CONFIG_DIR": str(yolo_dir)}
    if command([*pip, "check"], log, environment=environment):
        raise RuntimeError(f"pip check failed for {profile}; see {log.name}")
    result = command([str(python), "-c", IMPORT_SMOKES[profile]], log, timeout=120, environment=env)
    if result:
        raise RuntimeError(f"Import contract failed for {profile}; see {log.name}")
    ready.touch()
    freeze = subprocess.check_output([*pip, "list", "--format=freeze"], text=True, env=environment, timeout=60)
    with (run / "environment" / "dependencies.txt").open("a", encoding="utf-8") as handle:
        handle.write(f"\n[{profile} | {profile_digest}]\n{freeze}")
    return str(python), env
'''


In [ ]:
SOURCES['jobs.py'] = r'''"""Video decoding, native prediction export and worker task dispatch."""
import itertools
import math
import shutil
import subprocess
import time
from contextlib import contextmanager
from pathlib import Path

from backends import Detector, Tracker
from common import BOX, NAMES, atomic_path, digest, freeze_predictions, json_write, write_line


@contextmanager
def capture(path):
    import cv2
    video = cv2.VideoCapture(str(path))
    try:
        if not video.isOpened():
            raise ValueError(f"OpenCV could not open {path}")
        yield video
    finally:
        video.release()


def frames(video, maximum=None):
    for _ in itertools.count(1) if maximum is None else range(maximum):
        ok, frame = video.read()
        if not ok:
            break
        yield frame


def probe_video(path):
    import cv2
    with capture(path) as video:
        fps = float(video.get(cv2.CAP_PROP_FPS))
        nominal_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        count, size = 0, None
        for count, frame in enumerate(frames(video), 1):
            current = (frame.shape[1], frame.shape[0])
            if size is None:
                size = current
            if size != current:
                raise ValueError("Variable frame dimensions require separate research handling")
    if count < 1 or not math.isfinite(fps) or fps <= 0:
        raise ValueError(f"Cannot decode valid frames and FPS for {path}")
    if nominal_frames > 0 and count != nominal_frames:
        raise ValueError(f"Inconsistent decode: container {nominal_frames}, decoded {count}")
    return dict(clip=path.stem, file=path.name, path=str(path), sha256=digest(path), bytes=path.stat().st_size,
                width=size[0], height=size[1], frames=count, nominal_fps=fps,
                duration_s=count / fps, duration_basis="decoded frame count / nominal FPS")


def dataset_audit(root):
    records = []
    failures = []
    for name in NAMES:
        try:
            records.append(probe_video(root / name))
        except Exception as error:
            failures.append({"file": name, "error": f"{type(error).__name__}: {error}"})
    return records, failures


class Overlay:
    def __init__(self, process):
        self.process = process

    def draw(self, frame, objects, tracks):
        if not self.process:
            return
        import cv2

        for entity in objects:
            x1, y1, x2, y2 = [round(entity[key]) for key in BOX]
            color = (70, 120, 255) if entity["class"] == "ball" else (80, 230, 100)
            label = f"ID {entity['track_id']}" if tracks else f"{entity['class']} {entity['score']:.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, label, (x1, max(15, y1 - 3)), cv2.FONT_HERSHEY_SIMPLEX, .42, color, 1, cv2.LINE_AA)
        self.process.stdin.write(frame.tobytes())


@contextmanager
def overlay(path, video, enabled):
    if not enabled:
        yield Overlay(None)
        return
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg with libx264 is required for annotated video. Install it or disable SAVE_ANNOTATED_VIDEOS.")
    with atomic_path(path) as temporary, (path.parent / f".{path.stem}.ffmpeg.log").open("w+") as log:
        command = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-f", "rawvideo", "-pix_fmt", "bgr24",
                   "-s", f"{video['width']}x{video['height']}", "-r", str(video["nominal_fps"]), "-i", "pipe:0",
                   "-an", "-c:v", "libx264", "-preset", "veryfast", "-threads", "1",
                   "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2", "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(temporary)]
        proc = subprocess.Popen(command, stdin=subprocess.PIPE, stderr=log)
        try:
            yield Overlay(proc)
            proc.stdin.close()
            code = proc.wait(timeout=90)
            if code or temporary.stat().st_size == 0:
                log.seek(0)
                raise RuntimeError(f"Overlay encoder exit={code}: {log.read()}")
        except (BrokenPipeError, OSError) as error:
            log.seek(0)
            raise RuntimeError(f"Overlay encoder error: {log.read()}") from error
        finally:
            if proc.poll() is None:
                proc.kill()
                proc.wait()
            proc.stdin.close()
    (path.parent / f".{path.stem}.ffmpeg.log").unlink(missing_ok=True)


class JobRunner:
    def __init__(self, device):
        import cv2
        cv2.setNumThreads(1)
        self.device = "cpu" if device.startswith("cpu:") else device
        self.detector = None
        self.detector_error = None
        self.torch = None
        self.gpu_uuid = None
        if device.startswith("cuda:"):
            import torch
            torch.set_num_threads(1)
            torch.cuda.set_device(device)
            from environment import cuda_uuid
            self.torch = torch
            self.gpu_uuid = cuda_uuid(torch, torch.cuda.current_device())

    def sync(self):
        if self.torch:
            self.torch.cuda.synchronize(self.device)

    def memory(self):
        if not self.torch:
            return {}
        cuda = self.torch.cuda
        return dict(cuda_peak_allocated_bytes=cuda.max_memory_allocated(self.device),
                    cuda_peak_reserved_bytes=cuda.max_memory_reserved(self.device),
                    cuda_allocated_bytes=cuda.memory_allocated(self.device), cuda_reserved_bytes=cuda.memory_reserved(self.device))

    def __call__(self, task):
        operation = task["kind"]
        if operation == "detect":
            if self.detector_error:
                raise RuntimeError(f"Model initialization previously failed in this worker: {self.detector_error}")
            start = time.perf_counter()
            load_s = 0
            if self.detector is None:
                try:
                    self.detector = Detector(task["method"], task["asset"], self.device, task["precision"], task["score"])
                    self.sync()
                    load_s = time.perf_counter() - start
                except Exception as error:
                    self.detector_error = repr(error)
                    raise
            if self.torch:
                self.torch.cuda.reset_peak_memory_stats(self.device)
            return {**self.detect(task), **self.memory(), "model_load_s": load_s}
        if operation == "track":
            if self.torch:
                self.torch.cuda.reset_peak_memory_stats(self.device)
            return {**self.track(task), **self.memory()}
        if operation == "osnet":
            if self.torch:
                self.torch.cuda.reset_peak_memory_stats(self.device)
            from advanced import extract_osnet
            return {**extract_osnet(task, self.device), **self.memory()}
        raise ValueError(f"Unknown job kind: {operation}")

    def detect(self, task):
        import cv2
        video = task["video"]
        output = Path(task["output"])
        pred = output / "predictions" / f"{video['clip']}.jsonl"
        overlay_path = output / "annotated_videos" / f"{video['clip']}.mp4"
        expected = min(task["limit"] or video["frames"], video["frames"])
        count = boxes = batches = 0
        inference = 0.0
        with capture(video["path"]) as source, atomic_path(pred) as temporary, temporary.open("w") as rows, \
                overlay(overlay_path, video, task["overlays"]) as annotated:
            decoded = iter(frames(source, expected))
            while batch := list(itertools.islice(decoded, task["batch"])):
                rgb = [cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) for frame in batch]
                self.sync()
                start = time.perf_counter()
                results = self.detector.predict(rgb)
                self.sync()
                batches += 1
                inference += time.perf_counter() - start
                for frame, detections in zip(batch, results, strict=True):
                    count += 1
                    for i, d in enumerate(detections):
                        d["detection_id"] = f"{count}:{i}"
                    boxes += len(detections)
                    write_line(rows, {"frame": count, "detections": detections})
                    annotated.draw(frame, detections, False)
            if count != expected:
                raise ValueError(f"Decoded {count} frames during inference, expected {expected}")
        fingerprint = digest(pred)
        json_write(pred.with_suffix(".json"), {
            "clip": video["clip"], "clip_sha256": video["sha256"], "prediction_sha256": fingerprint,
            "frames": count, "complete_clip": count == video["frames"], "method": task["method"],
            "score_floor": task["score"], "asset": task["asset"], "model_contract": self.detector.load_info,
            "batch_size": task["batch"], "precision": task["precision"],
            "device": task.get("assigned_device", self.device), "worker_cuda_device": self.device,
            "gpu_uuid": self.gpu_uuid, "inference_batches": batches, "cuda_frames": count if self.torch else 0,
        })
        return {"frames": count, "boxes": boxes, "inference_s": inference, "inference_batches": batches,
                "cuda_frames": count if self.torch else 0,
                "clip_seconds": count / video["nominal_fps"], "prediction_path": str(pred),
                "prediction_sha256": fingerprint}

    def track(self, task):
        video = task["video"]
        output = Path(task["output"])
        raw = Path(task["cache"])
        pred = output / "tracks" / f"{video['clip']}.jsonl"
        mot = output / "tracks" / f"{video['clip']}.txt"
        overlay_path = output / "annotated_videos" / f"{video['clip']}.mp4"
        count = boxes = 0
        infer = 0.0
        start = time.perf_counter()
        tracker = Tracker(task["method"], video["nominal_fps"], task["thresholds"], task.get("asset"), self.device)
        self.sync()
        load_s = time.perf_counter() - start
        with capture(video["path"]) as source, atomic_path(pred) as json_tmp, json_tmp.open("w") as rows, \
                atomic_path(mot) as mot_tmp, mot_tmp.open("w") as mot_rows, \
                overlay(overlay_path, video, task["overlays"]) as annotated:
            for line in freeze_predictions(raw, video):
                ok, frame = source.read()
                if not ok:
                    raise ValueError("Frozen predictions outlast the decoded video")
                self.sync()
                start = time.perf_counter()
                tracks = tracker.update(line["detections"], frame)
                self.sync()
                infer += time.perf_counter() - start
                count = line["frame"]
                boxes += len(tracks)
                write_line(rows, {"frame": count, "tracks": tracks})
                for r in tracks:
                    mot_rows.write(f"{count},{r['track_id']},{r['x1'] + 1:.3f},{r['y1'] + 1:.3f},"
                                   f"{r['x2'] - r['x1']:.3f},{r['y2'] - r['y1']:.3f},{r['score']:.7f},-1,-1,-1\n")
                annotated.draw(frame, tracks, True)
        fingerprint = digest(raw)
        json_write(pred.with_suffix(".json"), {
            "clip": video["clip"], "clip_sha256": video["sha256"], "detection_sha256": fingerprint,
            "track_sha256": digest(pred), "frames": count, "method": task["method"], "parameters": tracker.parameters,
            "cache": str(raw), "asset": task.get("asset"),
        })
        return {"frames": count, "boxes": boxes, "inference_s": infer, "model_load_s": load_s,
                "clip_seconds": count / video["nominal_fps"], "prediction_path": str(pred), "cache_sha256": fingerprint}
'''


In [ ]:
SOURCES['schedule.py'] = r'''"""One spawn process per device, with a shared queue and explicit failure recovery."""
import argparse
import contextlib
import multiprocessing as mp
import os
import queue
import subprocess
import threading
import time
import traceback
from pathlib import Path

from common import Blocked, json_read, json_write, table
from jobs import JobRunner

SAMPLE_FIELDS = ("utc_epoch_s", "physical_index", "gpu_uuid", "utilization_pct", "memory_used_mib", "memory_total_mib")


def worker(device, first, work, events, log_path, factory, expected_uuid):
    pid = os.getpid()
    base = {"device": device, "pid": pid}
    last = time.monotonic()
    with open(log_path, "w", buffering=1) as log, contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
        try:
            runtime_device = device
            if device.startswith("cuda:"):
                logical = int(device.split(":")[1])
                visible = os.environ.get("CUDA_VISIBLE_DEVICES")
                token = visible.split(",")[logical].strip() if visible else str(logical)
                os.environ["CUDA_VISIBLE_DEVICES"] = f"GPU-{expected_uuid}" if expected_uuid else token
                runtime_device = "cuda:0"
            runner = factory(runtime_device)
            physical_uuid = getattr(runner, "gpu_uuid", None)
            if expected_uuid and physical_uuid != expected_uuid:
                raise RuntimeError(f"{device} worker resolved GPU {physical_uuid}, expected {expected_uuid}")
            if physical_uuid:
                base["gpu_uuid"] = physical_uuid
        except Exception:
            events.put(("startup_failure", dict(base, traceback=traceback.format_exc())))
            return
        idle = 0.0
        task = first
        while task is not None:
            start = time.monotonic()
            idle += start - last
            row = dict(base, id=task["id"], method=task["method"], clip=task.get("video", {}).get("clip", ""),
                       phase=task["phase"], start_monotonic_s=start)
            events.put(("start", row))
            try:
                metrics = runner({**task, "assigned_device": device})
                result = dict(row, state="EXECUTED", **metrics)
            except Exception as error:
                print(traceback.format_exc(), flush=True)
                state = "BLOCKED" if isinstance(error, Blocked) else "FAILED"
                result = dict(row, state=state, reason=f"{type(error).__name__}: {error}", traceback=traceback.format_exc())
            last = time.monotonic()
            events.put(("result", dict(result, wall_s=last - start, end_monotonic_s=last)))
            task = work.get()
        events.put(("worker_done", dict(base, idle_estimate_s=idle + time.monotonic() - last)))


def sample_devices(stop, samples, diagnostics):
    while not stop.is_set():
        try:
            result = subprocess.run(["nvidia-smi", "--query-gpu=index,uuid,utilization.gpu,memory.used,memory.total",
                                     "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5)
            if result.returncode:
                diagnostics.append(result.stderr.strip())
                return
            stamp = time.time()
            for line in result.stdout.splitlines():
                columns = [part.strip() for part in line.split(",")]
                if len(columns) == 5:
                    samples.append(dict(zip(SAMPLE_FIELDS, [stamp, *columns])))
        except (OSError, subprocess.TimeoutExpired) as error:
            diagnostics.append(str(error))
            return
        stop.wait(1)


def execute(tasks, devices, logdir, timeout, factory=JobRunner, telemetry=True, device_uuids=None):
    if not tasks or not devices:
        raise ValueError("Scheduler requires nonempty jobs and device plan")
    if len({job["id"] for job in tasks}) != len(tasks) or len(set(devices)) != len(devices):
        raise ValueError("Scheduler jobs and devices must have unique identifiers")
    logdir = Path(logdir)
    logdir.mkdir(parents=True, exist_ok=True)
    devices = devices[:len(tasks)]
    sorted_jobs = sorted(tasks, key=lambda job: (-job.get("cost", 0), job["id"]))
    context = mp.get_context("spawn")
    work, events = context.Queue(), context.Queue()
    processes = {}
    started = {}
    final = {}
    workers = {}
    events_by_id = {}
    samples, diagnostics = [], []
    stop = threading.Event()
    sampler = threading.Thread(target=sample_devices, args=(stop, samples, diagnostics), daemon=True)
    submissions = time.monotonic()
    if telemetry:
        sampler.start()

    def receive(message):
        kind, event = message
        pid = event["pid"]
        if kind == "start":
            started[pid] = event
            events_by_id[event["id"]] = event
            print(f"[{event['device']} | pid {pid}] {event['phase']} {event['method']} {event['clip']}", flush=True)
        elif kind == "result":
            started.pop(pid, None)
            final[event["id"]] = event
        elif kind in {"startup_failure", "worker_done"}:
            workers[pid].update(event)
            workers[pid]["terminal"] = kind

    try:
        # Seed one job per worker, then let available workers pull remaining jobs.
        for device, first in zip(devices, sorted_jobs):
            tag = device.replace(":", "_")
            log = logdir / f"{tag}.log"
            process = context.Process(target=worker, args=(device, first, work, events, str(log), factory, (device_uuids or {}).get(device)))
            process.start()
            processes[process.pid] = process
            workers[process.pid] = {"device": device, "pid": process.pid, "seed_job": first["id"], "log": str(log),
                                    "spawned": time.monotonic()}
        for job in sorted_jobs[len(devices):]:
            work.put(job)
        for _ in devices:
            work.put(None)

        while any(process.is_alive() for process in processes.values()):
            try:
                receive(events.get(timeout=0.15))
            except queue.Empty:
                pass
            now = time.monotonic()
            for pid, process in processes.items():
                worker_info = workers[pid]
                active = started.get(pid)
                age = now - (active["start_monotonic_s"] if active else worker_info["spawned"])
                if process.is_alive() and age > timeout and (active or worker_info["seed_job"] not in final):
                    worker_info["timeout"] = f"Worker exceeded {timeout}s for active job or process initialization"
                    process.terminate()
                    process.join(timeout=4)
                    if process.is_alive():
                        process.kill()
        for process in processes.values():
            process.join()
        while True:
            try:
                receive(events.get(timeout=.1))
            except queue.Empty:
                break
    finally:
        end = time.monotonic()
        stop.set()
        if telemetry:
            sampler.join(timeout=6)
        for pid, process in processes.items():
            if process.is_alive():
                process.kill()
                process.join(timeout=4)
            workers[pid]["exit_code"] = process.exitcode
        work.cancel_join_thread()
        work.close()
        events.close()

    initial = {info["seed_job"]: info for info in workers.values()}
    for job in tasks:
        if job["id"] in final:
            continue
        row = events_by_id.get(job["id"], {})
        assigned = workers.get(row.get("pid"), initial.get(job["id"]))
        reason = "All available workers stopped before returning this job"
        if assigned:
            lines = assigned.get("traceback", "").strip().splitlines()
            reason = assigned.get("timeout") or (lines[-1] if lines else f"Worker exited with code {assigned['exit_code']}")
        final[job["id"]] = {
            **row, "id": job["id"], "method": job["method"], "clip": job.get("video", {}).get("clip", ""),
            "phase": job["phase"], "state": "FAILED" if assigned else "BLOCKED", "reason": reason,
            "device": assigned["device"] if assigned else "", "pid": assigned["pid"] if assigned else "",
        }
    results = [final[job["id"]] for job in tasks]
    for pid, info in workers.items():
        successful = [row for row in results if row.get("pid") == pid and row["state"] == "EXECUTED"]
        info.update(frames=sum(r.get("frames", 0) for r in successful), successful_jobs=len(successful),
                    cuda_frames=sum(r.get("cuda_frames", 0) for r in successful),
                    inference_batches=sum(r.get("inference_batches", 0) for r in successful),
                    inference_s=sum(r.get("inference_s", 0) for r in successful),
                    failures=sum(r.get("pid") == pid and r["state"] != "EXECUTED" for r in results))
    if samples:
        table(logdir / "device_samples.csv", samples, SAMPLE_FIELDS)
    return dict(results=results, workers=list(workers.values()), wall_s=end - submissions,
                telemetry_samples=samples, telemetry_diagnostics=diagnostics)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("spec", type=Path)
    parser.add_argument("result", type=Path)
    args = parser.parse_args()
    spec = json_read(args.spec)
    result = execute(spec["tasks"], spec["devices"], spec["logdir"], spec["timeout"], telemetry=spec["telemetry"], device_uuids=spec.get("device_uuids"))
    json_write(args.result, result)


if __name__ == "__main__":
    main()
'''


In [ ]:
SOURCES['evaluate.py'] = r'''"""Evaluation receives reviewed annotations only after predictions are frozen."""
import argparse
from collections import defaultdict
from pathlib import Path

from common import BOX, Blocked, digest, freeze_predictions, json_lines, json_read, json_write

DETECTION_METRICS = ("precision50", "recall50", "mAP50", "mAP50_95", "ball_recall50", "small_player_recall50")
TRACKING_METRICS = ("HOTA", "IDF1", "ID_switches", "fragmentation")
REID_METRICS = ("rank1", "mAP", "false_match_rate", "same_team_rank1", "same_team_mAP")
IDENTITY_METRICS = ("reentry_accuracy", "false_merges", "false_splits", "identity_precision",
                    "identity_coverage", "recovery_delay_frames")


def bbox(row):
    x, y, right, bottom = [float(row[key]) for key in BOX]
    return [x, y, right - x, bottom - y]


def parse_reviewed(spec, input_by_clip, dense=False):
    annotations = json_read(spec)
    if annotations.get("schema") != "scoutra-reviewed/1" or annotations.get("human_reviewed") is not True:
        raise ValueError("GT requires schema scoutra-reviewed/1 and human_reviewed: true")
    scoped = {}
    for clip, item in annotations["clips"].items():
        if clip not in input_by_clip:
            continue
        video = input_by_clip[clip]
        if item["clip_sha256"] != video["sha256"]:
            raise ValueError(f"GT clip SHA-256 mismatch: {clip}")
        if dense and item.get("continuous_scene") is not True:
            raise ValueError(f"Dense identity metrics require one reviewed continuous scene per clip: {clip}")
        selected = {}
        for frame in item["frames"]:
            index = frame["frame"]
            if not isinstance(index, int) or index in selected or not 1 <= index <= video["frames"]:
                raise ValueError(f"Invalid or duplicated annotation frame {clip}/{index}")
            ids = set()
            for truth in frame["objects"]:
                x, y, w, h = bbox(truth)
                if not (x >= 0 and y >= 0 and w > 0 and h > 0 and x + w <= video["width"] and y + h <= video["height"]):
                    raise ValueError(f"Invalid annotated box at {clip}/{index}")
                if truth["class"] not in {"person", "player", "goalkeeper", "referee", "ball"}:
                    raise ValueError(f"Invalid class: {truth['class']!r}")
                if dense:
                    identity = truth.get("player_id")
                    if truth["class"] == "ball" or not identity or identity in ids:
                        raise ValueError(f"MOT labels must be people with unique physical IDs within frame {clip}/{index}")
                    ids.add(identity)
            selected[index] = frame["objects"]
        if dense and set(selected) != set(range(1, video["frames"] + 1)):
            raise ValueError(f"Tracking GT must exhaustively list each frame, including empty frames: {clip}")
        if selected:
            scoped[clip] = selected
    if not scoped:
        raise Blocked("No reviewed GT frames overlap with this benchmark workload")
    return scoped


def detection_metrics(predictions, reviewed, inputs, operating_point, min_score):
    import numpy as np
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    images, annotations, detections = [], [], []
    for clip, frames in reviewed.items():
        if clip not in predictions:
            continue
        video = inputs[clip]
        by_frame = {}
        for pred in freeze_predictions(predictions[clip], video):
            if pred["frame"] in frames:
                by_frame[pred["frame"]] = pred["detections"]
        if set(by_frame) != set(frames):
            raise ValueError(f"Benchmark predictions do not cover all annotated frames for {clip}")
        for frame, objects in frames.items():
            image_id = len(images) + 1
            images.append({"id": image_id, "width": video["width"], "height": video["height"]})
            for obj in objects:
                box = bbox(obj)
                annotations.append({"id": len(annotations) + 1, "image_id": image_id,
                                    "category_id": 2 if obj["class"] == "ball" else 1, "bbox": box,
                                    "area": box[2] * box[3], "iscrowd": 0, "role": obj["class"]})
            for pred in by_frame[frame]:
                detections.append({"image_id": image_id, "category_id": 2 if pred["class"] == "ball" else 1,
                                   "bbox": bbox(pred), "score": pred["score"]})
    if not annotations:
        raise Blocked("No positive reviewed GT boxes overlap with complete predictions")
    if operating_point < min_score:
        raise ValueError("Evaluation score threshold must be >= the detector cache score floor")
    categories = [{"id": 1, "name": "person"}, {"id": 2, "name": "ball"}]
    ground = COCO()
    ground.dataset = {"images": images, "annotations": annotations, "categories": categories, "info": {}}
    ground.createIndex()
    if detections:
        result = ground.loadRes(detections)
    else:
        result = COCO()
        result.dataset = {"images": images, "annotations": [], "categories": categories}
        result.createIndex()
    evaluation = COCOeval(ground, result, "bbox")
    evaluation.evaluate()
    evaluation.accumulate()
    confusion = defaultdict(lambda: [0, 0, 0])
    player_gt = {item["id"] for item in annotations if item["role"] == "player"}
    small_hits = small_eligible = 0
    all_area, small_area = map(tuple, evaluation.params.areaRng[:2])

    # COCO matches are confidence-descending. Lower scores cannot affect these TP/FP decisions.
    for entry in evaluation.evalImgs:
        if entry is None:
            continue
        mask = np.asarray(entry["dtScores"]) >= operating_point
        valid = mask & ~np.asarray(entry["dtIgnore"][0], dtype=bool)
        true = valid & (entry["dtMatches"][0] > 0)
        valid_gt = int(np.sum(~np.asarray(entry["gtIgnore"], dtype=bool)))
        area = tuple(entry["aRng"])
        counts = confusion[(entry["category_id"], area)]
        if entry["category_id"] == 1 and area == small_area:
            small_hits += sum(match in player_gt for match in entry["dtMatches"][0][true])
            small_eligible += sum(gid in player_gt and not ignore for gid, ignore in zip(entry["gtIds"], entry["gtIgnore"]))
        counts[0] += int(true.sum())
        counts[1] += int(valid.sum())
        counts[2] += valid_gt
    all_counts = np.sum([confusion[(c, all_area)] for c in (1, 2)], axis=0)

    def ratio(a, b):
        return float(a / b) if b else None

    def average(array):
        return float(array[array >= 0].mean()) if (array >= 0).any() else None

    ball = confusion[(2, all_area)]
    precision = evaluation.eval["precision"]
    return dict(precision50=ratio(all_counts[0], all_counts[1]), recall50=ratio(all_counts[0], all_counts[2]),
                mAP50=average(precision[0, :, :, 0, -1]), mAP50_95=average(precision[:, :, :, 0, -1]),
                ball_recall50=ratio(ball[0], ball[2]), small_player_recall50=ratio(small_hits, small_eligible))


def iou_matrix(left, right):
    import numpy as np

    a = np.array([[row[k] for k in BOX] for row in left], dtype=float).reshape(-1, 4)
    b = np.array([[row[k] for k in BOX] for row in right], dtype=float).reshape(-1, 4)
    overlap = np.maximum(np.minimum(a[:, None, 2:], b[None, :, 2:]) - np.maximum(a[:, None, :2], b[None, :, :2]), 0)
    intersection = np.prod(overlap, axis=2)
    union = np.prod(a[:, 2:] - a[:, :2], axis=1)[:, None] + np.prod(b[:, 2:] - b[:, :2], axis=1)[None, :] - intersection
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)


def checked_tracks(path, video):
    path = Path(path)
    sidecar = json_read(path.with_suffix(".json"))
    if sidecar["clip_sha256"] != video["sha256"] or sidecar["track_sha256"] != digest(path):
        raise ValueError("Track file fingerprint disagrees with its sidecar or source video")
    rows = list(json_lines(path))
    if len(rows) != sidecar["frames"] or any(i != r["frame"] for i, r in enumerate(rows, 1)):
        raise ValueError("Invalid dense tracking frame sequence")
    return rows


def tracking_metrics(predictions, reviewed, inputs):
    import numpy as np
    from trackeval.metrics import CLEAR, HOTA, Identity

    evaluators = [HOTA(), Identity({"THRESHOLD": .5, "PRINT_CONFIG": False}), CLEAR({"THRESHOLD": .5, "PRINT_CONFIG": False})]
    results = [dict() for _ in evaluators]
    for clip, frames in reviewed.items():
        if clip not in predictions:
            continue
        rows = checked_tracks(predictions[clip], inputs[clip])
        if len(rows) != len(frames):
            raise ValueError(f"MOT scoring requires predictions for every reviewed frame of {clip}")
        true = [frames[r["frame"]] for r in rows]
        predicted = [r["tracks"] for r in rows]

        def ids(frame_objects, field):
            mapping = {}
            arrays = [np.asarray([mapping.setdefault(obj[field], len(mapping)) for obj in row], dtype=int)
                      for row in frame_objects]
            return mapping, arrays

        gt_ids, truth = ids(true, "player_id")
        tracker_ids, estimates = ids(predicted, "track_id")
        data = dict(num_gt_ids=len(gt_ids), num_tracker_ids=len(tracker_ids), gt_ids=truth, tracker_ids=estimates,
                    num_gt_dets=sum(map(len, truth)), num_tracker_dets=sum(map(len, estimates)), num_timesteps=len(rows),
                    similarity_scores=[iou_matrix(g, p) for g, p in zip(true, predicted)])
        for metric, by_clip in zip(evaluators, results):
            by_clip[clip] = metric.eval_sequence(data)
    if not results[0]:
        raise Blocked("No complete tracking predictions overlap with the reviewed GT")
    hota, identity, clear = [metric.combine_sequences(by_clip) for metric, by_clip in zip(evaluators, results)]
    return dict(HOTA=float(hota["HOTA"].mean()), IDF1=float(identity["IDF1"]),
                ID_switches=int(clear["IDSW"]), fragmentation=int(clear["Frag"]))


def qualified(metrics):
    return {k: {"state": "EXECUTED" if value is not None else "NOT_EVALUATED", "value": value}
            for k, value in metrics.items()}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("spec", type=Path)
    parser.add_argument("output", type=Path)
    args = parser.parse_args()
    work = json_read(args.spec)
    dense = work["kind"] == "tracking"
    reviewed = parse_reviewed(work["annotations"], work["inputs"], dense)
    missing = set(reviewed) - set(work["predictions"])
    if missing:
        raise ValueError(f"One or more reviewed clips have no completed prediction: {sorted(missing)}")
    options = [] if dense else [work["operating_point"], work["cache_floor"]]
    handler = tracking_metrics if dense else detection_metrics
    result = handler(work["predictions"], reviewed, work["inputs"], *options)
    json_write(args.output, {"metrics": qualified(result), "gt_sha256": digest(work["annotations"]),
                            "reviewed_clips": sorted(set(reviewed) & set(work["predictions"]))})


def identity_metrics(tracks, mapping, video, annotations, reviewed):
    """Evaluate clip-local grouping using frozen predictions, reviewed anchors and re-entry episodes."""
    import numpy as np

    from scipy.optimize import linear_sum_assignment

    clip = video["clip"]
    rows = checked_tracks(tracks, video)
    if len(rows) != len(reviewed) or len(rows) != video["frames"]:
        raise ValueError(f"Identity scoring requires complete reviewed tracks: {clip}")
    group = mapping.get("groups")
    frame_groups = mapping.get("frame_groups", {})
    if mapping["tracking_sha256"] != digest(tracks) or mapping["clip_sha256"] != video["sha256"]:
        raise ValueError("Identity output does not match the tracking and clip hashes")

    alignment = {}
    groups_by_gt, gts_by_group = defaultdict(set), defaultdict(set)
    for row in rows:
        frame = row["frame"]
        gt = reviewed[frame]
        ts = row["tracks"]
        if gt and ts:
            matrix = iou_matrix(gt, ts)
            permitted = matrix >= 0.5
            # Maximize the number of threshold-valid edges before optimizing IoU.
            left, right = linear_sum_assignment(-(permitted * (matrix + 1 + min(matrix.shape))))
            for i, j in zip(left, right):
                if permitted[i, j]:
                    label = gt[i]["player_id"]
                    key = str(ts[j]["track_id"])
                    cluster = group.get(key) if group is not None else frame_groups.get(str(frame), {}).get(key)
                    if cluster is None:
                        continue
                    alignment[(frame, label)] = cluster
                    groups_by_gt[label].add(cluster)
                    gts_by_group[cluster].add(label)
    anchors = {}
    used_players = set()
    for row in annotations["anchors"]:
        frame, player = row["frame"], row["player_id"]
        if player in used_players or frame not in reviewed or player not in {x["player_id"] for x in reviewed[frame]}:
            raise ValueError("Each identity anchor must name a unique reviewed visible player")
        used_players.add(player)
        cluster = alignment.get((frame, player))
        if cluster is None:
            continue
        if cluster in anchors and anchors[cluster] != player:
            # Ambiguous anchors can arise after an incorrect upstream merge.
            anchors[cluster] = None
        else:
            anchors[cluster] = player

    visible = sum(len(rows) for rows in reviewed.values())
    assigned = sum(anchors.get(cluster) is not None for cluster in alignment.values())
    correct = sum(anchors.get(cluster) == player for (_, player), cluster in alignment.items())
    delays = []
    episodes = annotations.get("reentries", [])
    for item in episodes:
        player = item["player_id"]
        first, last = item["return_frame"], item["end_frame"]
        if player not in used_players or not 1 <= first <= last <= len(rows):
            raise ValueError("Each re-entry needs an earlier anchor and an in-bounds visible episode")
        origin = next(x["frame"] for x in annotations["anchors"] if x["player_id"] == player)
        before_return = {x["player_id"] for x in reviewed[first - 1]} if first > 1 else set()
        if origin >= first or player in before_return or player not in {x["player_id"] for x in reviewed[first]}:
            raise ValueError("Re-entry must follow a reviewed absent frame and begin with visible GT")
        for frame in range(first, last + 1):
            cluster = alignment.get((frame, player))
            if cluster is not None and anchors.get(cluster) == player:
                delays.append(frame - first)
                break
    return {
        "metrics": {
            "reentry_accuracy": len(delays) / len(episodes) if episodes else None,
            "recovery_delay_frames": float(np.median(delays)) if delays else None,
            "identity_precision": correct / assigned if assigned else None,
            "identity_coverage": assigned / visible if visible else None,
            "false_merges": sum(math_comb(len(ids)) for ids in gts_by_group.values()) if visible else None,
            "false_splits": sum(math_comb(len(ids)) for ids in groups_by_gt.values()) if visible else None,
        },
        "counts": {"anchors": len(annotations["anchors"]), "ambiguous_anchors": sum(x is None for x in anchors.values()),
                   "gt_person_frames": visible, "iou_matched_person_frames": len(alignment), "assigned_person_frames": assigned,
                   "correct_person_frames": correct, "return_episodes": len(episodes), "recoveries": len(delays)},
        "recovery_delays_frames": delays,
        "contaminated_groups": {str(group): sorted(map(str, ids)) for group, ids in gts_by_group.items() if len(ids) > 1},
        "split_identities": {str(player): sorted(ids) for player, ids in groups_by_gt.items() if len(ids) > 1},
    }


def math_comb(n):
    return n * (n - 1) // 2


if __name__ == "__main__":
    main()
'''


In [ ]:
SOURCES['advanced.py'] = r'''"""Opt-in independent research methods and methods with recorded prerequisites."""
import csv
import importlib.util
import itertools
import math
import sys
import time
from collections import defaultdict
from pathlib import Path

from common import Blocked, atomic_path, digest, fetch, freeze_predictions, json_read, json_write, verified
from evaluate import checked_tracks

AGENTCOACH_COMMIT = "eef7b0c5d757844fbbb1eeafa6a8050a863a6623"
AGENTCOACH_HASH = "2a9bd53fb03db7557991d1fa30bb387ee3f9fe086f52f13113f6235fb024f148"
PART_COMMITS = {
    "prtreid_global": ("prtreid", "30617a75967e84d5d516959c4b84cbeea6f56493", 8),
    "prtreid_parts": ("prtreid", "30617a75967e84d5d516959c4b84cbeea6f56493", 8),
    "bpbreid_parts": ("torchreid", "a2dc4304284784d8b2c061764512f0698aa38c21", 6),
}
ADVANCED = {
    "detection": {
        "specialized_soccer_ball": "No licensed, source-pinned ball checkpoint with verified class map was selected. Generic COCO sports ball is available in the core detectors.",
    },
    "tracking": {
        "deep_eiou": None,
        "mcbyteplusplus": "The pinned upstream tracker requires EdgeTAM masks, the sports Re-ID checkpoint and a validated frozen-detection bridge in its separate Torch/CUDA environment.",
    },
    "reid": dict.fromkeys(("prepare_crops", "osnet", *PART_COMMITS), None),
    "identity": {
        "no_link": None, "agentcoach_reid_only": None, "cosine_tracklet_link": None,
        "gta_link": None,
        "gtatrack": "Separate GTATrack source, checkpoints and exact detector-feature contract have not been validated.",
        "mcbyte_identity_bank": "Requires real McByte++ association and EdgeTAM mask streams, bank-state export and the upstream bank-on/off switch.",
        "cross_multiplication_reid": "No verified standalone original checkpoint and inference/preprocessing contract is wired.",
    },
    "jersey": {
        "parseq": "The PARSeq STR runtime and jersey-specific weights/tokenizer have not been validated in an isolated profile.",
        "koshkina": "Needs compatible legibility, pose, torso-crop and PARSeq weights plus the upstream tracklet workflow across its environments.",
        "temporal_ocr_vote": None,
    },
    "calibration": {
        "opencv_homography": None, "tvcalib": "Needs pinned pitch-segment network/checkpoint, pitch model and calibration runner.",
        "pnlcalib": "Needs pinned line/point detectors, checkpoint hashes and verified correspondence format.",
    },
    "ball": {
        "temporal_ball_baseline": None, "ballradar": "Needs verified upstream expected ball/player trajectories and checkpoint input conversion.",
        "pathcrf": "Exact source/checkpoint and candidate-association contract are unverified.",
    },
    "events": {
        "t_deed": "Needs checkpoint-specific frame rate, backbone/features and SoccerNet event alignment.",
        "opensportslib": "No committed real event backend and usable checkpoint resolved for this name.",
        "footpass": "Requires verified checkpoint, pass taxonomy and continuous time/actor labels.",
    },
    "vlm": {
        "qwen3_vl": "Needs standalone model revision, video sampling, T4-compatible precision/kernels and measured memory fit.",
        "qwen3_5": "No verified 27B–35B T4 fit, quantization or multi-device execution plan.",
        "soccerchat": "Requires exact base VLM, adapters, video input contract and checkpoint/license verification.",
        "football_qwen": "No selected auditable football-specific model revision with a verified T4-compatible runtime.",
    },
}


def crop_manifest(path):
    path = Path(path)
    records = []
    with path.open(newline="", encoding="utf-8") as handle:
        for item in csv.DictReader(handle):
            image = verified(path.parent / item["path"], item["sha256"])
            records.append({**item, "path": str(image)})
    ids = [row["crop_id"] for row in records]
    if not ids or any(not i for i in ids) or len(ids) != len(set(ids)):
        raise ValueError("Crop manifest needs nonempty unique crop_id and verified crop bytes")
    return records


def osnet_backend(asset, device):
    import torch
    from boxmot.reid.core.auto_backend import ReidAutoBackend

    path = Path(asset["path"])
    backend = ReidAutoBackend(weights=path, device=torch.device(device), half=False).model
    verify_osnet_backbone(backend, path, device)
    return backend


def verify_osnet_backbone(backend, path, device):
    import torch

    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    weights = {k.removeprefix("module."): v for k, v in checkpoint.get("state_dict", checkpoint).items()}
    loaded = backend.model.state_dict()
    missing = [k for k, v in loaded.items() if not k.startswith("classifier.") and
               (k not in weights or not torch.equal(v.detach().cpu(), weights[k]))]
    if missing:
        raise ValueError(f"OSNet backbone tensors do not match the supplied checkpoint: {missing[:10]}")
    if next(backend.model.parameters()).device != torch.device(device):
        raise RuntimeError("OSNet model is not on the assigned device")


def extract_osnet(task, device):
    import cv2
    import numpy as np
    import torch

    crops = crop_manifest(task["crops"])
    start = time.perf_counter()
    model = osnet_backend(task["asset"], device)
    sync = lambda: torch.cuda.synchronize(device) if device.startswith("cuda") else None
    sync()
    load_s = time.perf_counter() - start
    all_vectors = []
    inference = 0.0
    batch = task["batch"]
    for offset in range(0, len(crops), batch):
        blocks = []
        for crop in crops[offset:offset + batch]:
            image = cv2.imread(crop["path"])
            if image is None:
                raise ValueError(f"Cannot open crop {crop['path']}")
            box = np.asarray([[0, 0, image.shape[1], image.shape[0]]])
            blocks.append(model.get_crops(box, image))
        sync()
        start = time.perf_counter()
        with torch.inference_mode():
            vector = model.inference_postprocess(model.forward(model.inference_preprocess(torch.cat(blocks))))
        sync()
        inference += time.perf_counter() - start
        all_vectors.append(np.asarray(vector))
    array = np.concatenate(all_vectors)
    write_embeddings(task["output"], crops, {"global": array}, task["asset"], {"input_size": model.input_shape,
                     "color": "RGB", "resize": "OpenCV INTER_LINEAR via BoxMOT", "batch": batch},
                     source_manifest=task["crops"])
    return {"crops_processed": len(crops), "model_load_s": load_s, "inference_s": inference,
            "prediction_path": str(task["output"])}


def write_embeddings(path, crops, features, asset, preprocessing, source_manifest):
    import numpy as np

    path = Path(path)
    count = len(crops)
    if any(x.shape[0] != count or not np.isfinite(x).all() for x in features.values()):
        raise ValueError("Nonfinite embeddings or a misaligned crop batch")
    if "parts" in features:
        if features["parts"].ndim != 3 or features["visibility"].shape != features["parts"].shape[:2]:
            raise ValueError("Expected full parts [N,P,D] and visibility [N,P]")
        if features["part_masks"].shape[:2] != features["parts"].shape[:2] or features["part_masks"].ndim != 4:
            raise ValueError("Expected full spatial part_masks [N,P,H,W]")
    if "global" in features and (features["global"].ndim != 2 or not np.linalg.norm(features["global"], axis=1).all()):
        raise ValueError("Expected nonzero global vectors [N,D]")
    with atomic_path(path) as temporary:
        np.savez_compressed(temporary, crop_id=np.asarray([r["crop_id"] for r in crops]), **features)
    json_write(path.with_suffix(".json"), {"asset": asset, "preprocessing": preprocessing,
                                           "crop_manifest_sha256": digest(source_manifest),
                                           "embedding_sha256": digest(path), "crops": crops,
                                           "shapes": {k: list(v.shape) for k, v in features.items()}})


def vectors(path):
    import numpy as np

    path = Path(path)
    meta = json_read(path.with_suffix(".json"))
    if meta["embedding_sha256"] != digest(path):
        raise ValueError(f"Embedding SHA-256 mismatch: {path}")
    with np.load(path, allow_pickle=False) as source:
        data = {key: source[key] for key in source.files}
    if [x["crop_id"] for x in meta["crops"]] != data["crop_id"].tolist():
        raise ValueError("Embedding crop provenance mismatch")
    return data, meta


def pairwise_distance(data, query, gallery, visibility_min):
    import numpy as np

    if "parts" not in data:
        array = data["global"]
        left, right = array[query], array[gallery]
        left = left / np.linalg.norm(left, axis=-1, keepdims=True)
        right = right / np.linalg.norm(right, axis=-1, keepdims=True)
        return 1 - np.clip(left @ right.T, -1, 1)

    # Named analysis condition, separate from the upstream training/gallery evaluator.
    # Average cosine distance over jointly visible parts. No common visible part is unscorable.
    parts, visibility = data["parts"], data["visibility"]
    score = np.zeros((len(query), len(gallery)))
    count = np.zeros(score.shape, dtype=int)
    for index in range(parts.shape[1]):
        array = parts[:, index]
        norm = np.linalg.norm(array, axis=-1)
        normalized = array / np.maximum(norm[:, None], 1e-12)
        visible = (visibility[:, index] >= visibility_min) & (norm > 0)
        valid = visible[query, None] & visible[gallery][None, :]
        score += (1 - np.clip(normalized[query] @ normalized[gallery].T, -1, 1)) * valid
        count += valid
    return np.divide(score, count, out=np.full(score.shape, np.nan), where=count > 0)


def compare_embeddings(path, predictions, reviewed_labels, max_crops, visibility_min, threshold):
    import numpy as np

    data, meta = vectors(predictions)
    count = len(meta["crops"])
    rng = np.random.default_rng(1729)
    sample = np.sort(rng.choice(count, min(count, max_crops), replace=False))
    matrix = pairwise_distance(data, sample, sample, visibility_min)
    upper = matrix[np.triu_indices(len(matrix), 1)]
    valid = upper[np.isfinite(upper)]
    output = {
        "embedding_sha256": digest(predictions), "sample_crops": [meta["crops"][i]["crop_id"] for i in sample],
        "distance": "joint_visible_parts_mean_cosine" if "parts" in data else "global_cosine",
        "visibility_min": visibility_min, "sampled_pairs": int(len(upper)), "scorable_pairs": int(len(valid)),
        "histogram": np.histogram(np.clip(valid, 0, 2), bins=20, range=(0, 2))[0].tolist(),
        "histogram_edges": np.linspace(0, 2, 21).tolist(),
    }
    if reviewed_labels:
        output.update(gt_sha256=digest(reviewed_labels), metrics=retrieval(data, meta["crops"], reviewed_labels,
                                                                         visibility_min, threshold))
    json_write(path, output)
    return output


def retrieval(data, crops, labels, visibility_min, threshold):
    import numpy as np

    labels = json_read(labels)
    if labels.get("human_reviewed") is not True:
        raise ValueError("Re-ID labels must be human reviewed")
    index = {row["crop_id"]: i for i, row in enumerate(crops)}
    roles = {"query": [], "gallery": []}
    seen = set()
    for row in labels["crops"]:
        key = row["crop_id"]
        if key not in index or key in seen or crops[index[key]]["sha256"] != row["sha256"]:
            raise ValueError("Re-ID labels must identify unique supplied crops by ID and SHA-256")
        if any(not row.get(k) for k in ("match_id", "player_id", "team_id", "tracklet_id")):
            raise ValueError("Re-ID labels require match, physical player, team and original tracklet IDs")
        seen.add(key)
        roles[row["split"]].append(row)
    if not all(roles.values()):
        raise ValueError("Both query and gallery are required")

    gallery = roles["gallery"]
    gallery_indices = [index[x["crop_id"]] for x in gallery]
    all_metrics, team_metrics = [], []
    false = negatives = 0
    for query in roles["query"]:
        qi = index[query["crop_id"]]
        distance = pairwise_distance(data, [qi], gallery_indices, visibility_min)[0]
        eligible = np.asarray([p["match_id"] == query["match_id"] and p["tracklet_id"] != query["tracklet_id"]
                               and p["sha256"] != query["sha256"] for p in gallery]) & np.isfinite(distance)
        same_id = np.asarray([p["player_id"] == query["player_id"] for p in gallery])
        same_team = np.asarray([p["team_id"] == query["team_id"] for p in gallery])
        if threshold is not None:
            negative = eligible & ~same_id
            negatives += int(negative.sum())
            false += int((negative & (distance <= threshold)).sum())
        ordering = np.argsort(distance, kind="stable")
        for scope, target in ((eligible, all_metrics), (eligible & same_team, team_metrics)):
            relevant = same_id[ordering[scope[ordering]]]
            if relevant.any():
                ranks = np.flatnonzero(relevant) + 1
                target.append((float(relevant[0]), float((np.arange(1, len(ranks) + 1) / ranks).mean())))

    def metric(rows, column):
        return float(np.mean(rows, axis=0)[column]) if rows else None

    return dict(rank1=metric(all_metrics, 0), mAP=metric(all_metrics, 1),
                same_team_rank1=metric(team_metrics, 0), same_team_mAP=metric(team_metrics, 1),
                false_match_rate=false / negatives if negatives else None, valid_queries=len(all_metrics),
                valid_same_team_queries=len(team_metrics), negative_pairs=negatives, distance_threshold=threshold)


def identity_output(method, tracks, video, output, features=None, module=None, threshold=None):
    import numpy as np

    rows = checked_tracks(tracks, video)
    track_hash = digest(tracks)
    ids = sorted({entry["track_id"] for row in rows for entry in row["tracks"]})
    if not ids:
        raise Blocked(f"No local tracks for {video['clip']}")
    groups = {i: str(i) for i in ids}
    evidence = []
    if method != "no_link":
        if not features:
            raise Blocked("Identity linking requires global embeddings for the same frozen tracklets")
        data, meta = vectors(features)
        if "global" not in data:
            raise Blocked("This comparison requires global embeddings")
        lookup = defaultdict(list)
        for vector, crop in zip(data["global"], meta["crops"]):
            if crop.get("clip") == video["clip"] and crop.get("tracking_sha256") == track_hash:
                lookup[int(crop["track_id"])].append(vector)
        if not lookup:
            raise Blocked("No global crops are tied to this exact clip and track-prediction SHA-256")
        by_frame = [{x["track_id"] for x in r["tracks"]} for r in rows]
        cooccurring = {pair for group in by_frame for pair in itertools.combinations(sorted(group), 2)}
        if method == "agentcoach_reid_only":
            metadata = [module.TrackMeta(track_id=i, team="unknown", role="player", jersey_number=None, jersey_confidence=0,
                                         frame_count=sum(i in frame for frame in by_frame),
                                         reid_centroid=module.compute_centroid(lookup[i]) if lookup[i] else None)
                        for i in ids]
            mapped, clusters = module.consolidate(metadata, same_person_threshold=threshold,
                                                  orphan_absorb_threshold=max(threshold, 0.92), cooccur_pairs=cooccurring)
            groups = {i: str(mapped[i]) for i in ids}
            evidence = [row.to_dict() for row in clusters]
        else:
            centroid = {i: np.mean(v, axis=0) for i, v in lookup.items() if v}
            score = []
            for a, b in itertools.combinations(sorted(centroid), 2):
                x, y = centroid[a], centroid[b]
                norm = np.linalg.norm(x) * np.linalg.norm(y)
                if norm > 0:
                    score.append((float(np.clip(np.dot(x, y) / norm, -1, 1)), a, b))
            for similarity, a, b in sorted(score, reverse=True):
                left = [i for i in ids if groups[i] == groups[a]]
                right = [i for i in ids if groups[i] == groups[b]]
                decision = groups[a] != groups[b] and similarity >= threshold and not any(
                    tuple(sorted((i, j))) in cooccurring for i in left for j in right)
                evidence.append(dict(track_a=a, track_b=b, cosine=similarity, linked=decision))
                if decision:
                    label = groups[a]
                    groups.update(dict.fromkeys(right, label))
    json_write(output, {"method": method, "clip_sha256": video["sha256"], "tracking_sha256": track_hash,
                        "source_commit": AGENTCOACH_COMMIT if module else None, "threshold": threshold,
                        "offline": True, "groups": groups, "evidence": evidence})
    return len(ids)


def load_agentcoach(cache, online):
    base = f"https://raw.githubusercontent.com/outstandingcandy/AgentCoach/{AGENTCOACH_COMMIT}"
    path = fetch(f"{base}/goalinsight/track_consolidation/_consolidator.py",
                 cache / "agentcoach_consolidator.py", AGENTCOACH_HASH, online)
    spec = importlib.util.spec_from_file_location("scoutra_agentcoach_upstream", path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module


def homography(video, spec, output):
    import cv2
    import numpy as np

    labels = json_read(spec)
    if labels.get("human_reviewed") is not True:
        raise ValueError("Pitch landmarks require human review")
    item = labels["clips"].get(video["clip"])
    if not item:
        raise Blocked(f"No pitch landmarks for {video['clip']}")
    if item["clip_sha256"] != video["sha256"]:
        raise ValueError("Pitch landmark video hash mismatch")
    results = []
    for row in item["frames"]:
        image, world = [np.asarray(row[field], dtype=np.float64) for field in ("pixel_xy", "pitch_xy_m")]
        count = len(image)
        if image.shape != world.shape or image.shape != (count, 2) or count < 4 or not 1 <= row["frame"] <= video["frames"]:
            raise ValueError("Supply four or more pixel/meter correspondences for each valid frame")
        mat, mask = cv2.findHomography(image, world, cv2.RANSAC, 0.5)
        if mat is None or np.linalg.matrix_rank(mat) != 3 or int(mask.sum()) < 4:
            raise ValueError("Degenerate landmarks or insufficient inliers")
        meters = cv2.perspectiveTransform(image.reshape(-1, 1, 2), mat)[:, 0]
        error = np.linalg.norm(meters - world, axis=1)
        results.append(dict(frame=row["frame"], H_image_to_pitch=mat.tolist(), inliers=mask[:, 0].tolist(),
                            fit_rmse_m=float(np.sqrt(np.mean(error[mask[:, 0] == 1] ** 2)))))
    if not results:
        raise Blocked("No frame-specific pitch correspondences")
    json_write(output, {"clip_sha256": video["sha256"], "source_sha256": digest(spec), "frames": results})
    return len(results)


def temporal_ocr(input_path, output):
    rows = json_read(input_path)
    by_track = defaultdict(lambda: defaultdict(float))
    for row in rows:
        value = row["digits"]
        confidence = float(row["confidence"])
        if value.isascii() and value.isdigit() and 1 <= len(value) <= 2 and 0 <= confidence <= 1:
            by_track[(row["clip"], int(row["track_id"]))][value] += confidence
    if not by_track:
        raise Blocked("No valid jersey OCR candidates to aggregate")
    result = []
    for (clip, track_id), scores in sorted(by_track.items()):
        ranked = sorted(scores, key=lambda k: (-scores[k], k))
        top = ranked[0]
        if scores[top] > 0:
            result.append({"clip": clip, "track_id": track_id, "digits": top,
                           "vote_share": scores[top] / sum(scores.values()), "ties": sum(x == scores[top] for x in scores.values()),
                           "votes": dict(scores)})
    if not result:
        raise Blocked("All candidate votes have zero weight")
    json_write(output, {"source_sha256": digest(input_path), "votes": result})
    return len(result)


def temporal_ball(cache, video, output, min_score, gate_fraction, max_gap):
    records = []
    previous = newer = None
    gate = gate_fraction * math.hypot(video["width"], video["height"])
    for row in freeze_predictions(cache, video):
        idx = row["frame"]
        all_balls = [d for d in row["detections"] if d["class"] == "ball" and d["score"] >= min_score]
        if newer and idx - newer[0] > max_gap:
            previous = newer = None

        def center(box):
            return ((box["x1"] + box["x2"]) / 2, (box["y1"] + box["y2"]) / 2)

        prediction = newer[1] if newer else None
        if previous:
            elapsed = (idx - newer[0]) / (newer[0] - previous[0])
            prediction = tuple(a + elapsed * (a - b) for a, b in zip(newer[1], previous[1]))
        candidates = [b for b in all_balls if prediction is None or math.dist(center(b), prediction) <= gate]
        chosen = (min(candidates, key=lambda x: math.dist(center(x), prediction) / gate + (1 - x["score"]))
                  if candidates and prediction else max(candidates, key=lambda x: x["score"], default=None))
        if chosen:
            previous, newer = newer, (idx, center(chosen))
        records.append({"frame": idx, "state": "observed" if chosen else "missing", "detection": chosen})
    json_write(output, {"clip_sha256": video["sha256"], "detection_sha256": digest(cache),
                        "minimum_confidence": min_score, "gate_pixels": gate, "max_gap_frames": max_gap, "frames": records})
    return len(records)


def prepare_crops(cache_lookup, videos, output, stride, per_track):
    import cv2

    from common import table
    from jobs import capture

    records = []
    for video in videos:
        clip = video["clip"]
        tracks = cache_lookup(clip)
        tracking_hash = digest(tracks)
        seen, kept = defaultdict(int), defaultdict(int)
        with capture(video["path"]) as source:
            for row in checked_tracks(tracks, video):
                success, frame = source.read()
                if not success:
                    raise ValueError(f"Source ended before track cache for {clip}")
                for person in row["tracks"]:
                    track = person["track_id"]
                    seen[track] += 1
                    if (seen[track] - 1) % stride or kept[track] >= per_track:
                        continue
                    x1, y1 = (int(math.floor(person[k])) for k in ("x1", "y1"))
                    x2, y2 = (int(math.ceil(person[k])) for k in ("x2", "y2"))
                    if x2 - x1 < 8 or y2 - y1 < 16:
                        continue
                    name = f"{clip}_{row['frame']:06}_{track}"
                    relative = Path("images") / f"{name}.png"
                    with atomic_path(output.parent / relative) as temporary:
                        if not cv2.imwrite(str(temporary), frame[y1:y2, x1:x2]):
                            raise RuntimeError(f"Crop serialization failed: {name}")
                    kept[track] += 1
                    records.append(dict(crop_id=name, path=str(relative), sha256=digest(output.parent / relative),
                                        clip=clip, frame=row["frame"], track_id=track, tracking_sha256=tracking_hash,
                                        clip_sha256=video["sha256"], source_detection_id=person["detection_id"]))
    if not records:
        raise Blocked("No crops met the immutable selection rules (track observations, stride, minimum 8×16 pixels)")
    table(output, records, tuple(records[0]))
    return len(records)


def identity_overlay(tracks, mapping_path, video, path):
    from jobs import capture, overlay

    mapping = json_read(mapping_path)
    groups = mapping.get("groups")
    frame_groups = mapping.get("frame_groups", {})
    with capture(video["path"]) as source, overlay(path, video, True) as annotated:
        for row in checked_tracks(tracks, video):
            ok, image = source.read()
            if not ok:
                raise ValueError("Clip ended before the identity output")
            labels = groups if groups is not None else frame_groups.get(str(row["frame"]), {})
            boxes = []
            for track in row["tracks"]:
                group = labels.get(str(track["track_id"]))
                value = group if group is not None else f"unassigned/{track['track_id']}"
                boxes.append({**track, "track_id": value})
            annotated.draw(image, boxes, True)
'''


In [ ]:
SOURCES['legacy.py'] = r'''"""Launch genuine upstream PRTReID/BPBReID in a user-provisioned isolated Python."""
import argparse
import os
import subprocess
import sys
import time
from pathlib import Path

from advanced import PART_COMMITS, crop_manifest, write_embeddings
from common import Blocked, json_read, verified
from environment import NO_NET_ENV


GTA_COMMIT = "e4d5cc4065ceb1ec3fa9dc7478455f13a8d7f9ca"
GTA_WEIGHT_HASH = "8d5b2fd8763db34c2aad69810466adf413f0426d9f8119d322227e0e639c5fbd"


def validate_checkout(entry, commit, name):
    executable = Path(entry["python"])
    repo = Path(entry["repo"]).resolve()
    if not executable.is_absolute() or not executable.is_file() or not (repo / ".git").exists():
        raise Blocked("Legacy Python and checkout must already exist in the isolated environment.")
    if os.path.abspath(executable) == os.path.abspath(sys.executable):
        raise Blocked("Use a separate Python environment for legacy Research Re-ID.")
    head = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True, timeout=20).strip()
    dirty = subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain", "--untracked-files=no"],
                                   text=True, timeout=20).strip()
    if head != commit or dirty:
        raise ValueError(f"Expected clean {name} checkout at {commit}; found {head}")
    return str(executable), repo


def legacy_environment(device):
    original = os.environ.get("CUDA_VISIBLE_DEVICES")
    index = int(device.split(":")[1])
    visible = original.split(",")[index].strip() if original else str(index)
    return {**os.environ, **NO_NET_ENV, "CUDA_VISIBLE_DEVICES": visible}


def part_profile(method, configuration, device, crops, output, batch=16):
    package, commit, outputs = PART_COMMITS[method]
    entry = configuration.get(package)
    if not entry:
        raise Blocked(f"Configure LEGACY_REID[{package!r}] with the exact source, compatible interpreter, config, checkpoint and HRNet assets.")
    required = {"python", "repo", "checkpoint", "checkpoint_sha256", "config", "config_sha256", "backbone",
                "backbone_sha256", "num_classes", "rights_reviewed", "provenance"}
    if not required <= set(entry) or entry["rights_reviewed"] is not True or not entry["provenance"]:
        raise Blocked(f"Legacy {package} requires a completed source/weight review and fields {sorted(required)}.")
    executable, repo = validate_checkout(entry, commit, package)
    paths = {key: str(verified(entry[key], entry[f"{key}_sha256"])) for key in ("checkpoint", "config", "backbone")}
    config = {**entry, **paths, "package": package, "method": method, "upstream_output_count": outputs, "upstream": commit,
              "repo": str(repo), "logical_device": device, "crops": crops, "output": str(output), "batch": batch}
    return executable, config, legacy_environment(device)


def parts(spec):
    import importlib
    import numpy as np
    import torch
    from yacs.config import CfgNode

    if not torch.cuda.is_available():
        raise RuntimeError("Legacy environment has no functional CUDA device")
    device = "cuda:0"
    torch.cuda.set_device(device)
    torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()
    sys.path.insert(0, spec["repo"])
    namespace = spec["package"]
    package = importlib.import_module(namespace)
    if not Path(spec["repo"]) in Path(package.__file__).resolve().parents:
        raise RuntimeError("Legacy import resolved outside the verified upstream checkout")
    extractor = importlib.import_module(f"{namespace}.tools.feature_extractor").FeatureExtractor
    constants_module = importlib.import_module(f"{namespace}.utils.constants")
    with open(spec["config"], encoding="utf-8") as f:
        cfg = CfgNode.load_cfg(f)
    cfg.defrost()
    cfg.model.bpbreid.hrnet_pretrained_path = spec["backbone"]
    kwargs = {"cfg": cfg, "image_size": (cfg.data.height, cfg.data.width), "device": device,
              "pixel_mean": cfg.data.norm_mean, "pixel_std": cfg.data.norm_std,
              "num_classes": spec["num_classes"], "verbose": False}
    model = extractor(**kwargs)
    # The original convenience loader silently skips incompatible keys. Require exact tensor coverage.
    try:
        checkpoint = torch.load(spec["checkpoint"], map_location="cpu", weights_only=False)
    except TypeError as error:
        if "weights_only" not in str(error):
            raise
        checkpoint = torch.load(spec["checkpoint"], map_location="cpu")
    state = checkpoint.get("state_dict", checkpoint)
    state = {k[7:] if k.startswith("module.") else k: v for k, v in state.items()}
    model.model.load_state_dict(state, strict=True)
    ckpt_metadata = {key: str(checkpoint[key]) for key in ("epoch", "arch") if key in checkpoint}
    torch.cuda.synchronize(device)
    load_s = time.perf_counter() - started
    manifest = crop_manifest(spec["crops"])
    batches = {}
    elapsed = 0.0
    for index in range(0, len(manifest), spec["batch"]):
        names = [x["path"] for x in manifest[index:index + spec["batch"]]]
        torch.cuda.synchronize(device)
        start = time.perf_counter()
        result = model(names)
        torch.cuda.synchronize(device)
        elapsed += time.perf_counter() - start
        if len(result) != spec["upstream_output_count"]:
            raise ValueError("Legacy method returned unexpected feature tuple")
        emb, visibility, masks = result[0], result[1], result[-1]
        if spec["method"].endswith("_parts"):
            key = constants_module.PARTS
            features = {"parts": emb[key], "visibility": visibility[key], "part_masks": masks[key]}
        else:
            features = {"global": emb[constants_module.GLOBAL]}
        for key, data in features.items():
            batches.setdefault(key, []).append(data.detach().cpu().numpy())
    asset = {k: spec[k] for k in ("upstream", "config_sha256", "checkpoint_sha256", "backbone_sha256", "provenance")}
    write_embeddings(spec["output"], manifest, {k: np.concatenate(v) for k, v in batches.items()},
                     {**asset, "checkpoint_metadata": ckpt_metadata},
                     {"input_size": kwargs["image_size"], "pixel_mean": kwargs["pixel_mean"], "pixel_std": kwargs["pixel_std"],
                      "torch": torch.__version__, "python": sys.version, "device": spec["logical_device"],
                      "model_load_s": load_s, "inference_s": elapsed, "crop_batch": spec["batch"],
                      "peak_cuda_allocated_bytes": torch.cuda.max_memory_allocated(device),
                      "peak_cuda_reserved_bytes": torch.cuda.max_memory_reserved(device)}, source_manifest=spec["crops"])


def gta_profile(configuration, device, tracks, video, output):
    required = {"python", "repo", "rights_reviewed", "provenance"}
    if not required <= set(configuration) or configuration["rights_reviewed"] is not True or not configuration["provenance"]:
        raise Blocked("Set GTA_CONFIG for a verified isolated GTA-Link checkout and sports checkpoint rights.")
    executable, repo = validate_checkout(configuration, GTA_COMMIT, "GTA-Link")
    checkpoint = verified(repo / "reid_checkpoints" / "sports_model.pth.tar-60", GTA_WEIGHT_HASH)
    parameters = {"eps": 0.7, "max_k": 3, "min_samples": 10, "min_len": 100,
                  "spatial_factor": 1.0, "merge_dist_thres": 0.4}
    configured = configuration.get("parameters", {})
    if not set(configured) <= parameters.keys():
        raise ValueError("Unrecognized GTA-Link parameter")
    parameters.update(configured)
    return executable, {
        "method": "gta_link", "repo": str(repo), "checkpoint": str(checkpoint), "tracks": str(tracks),
        "video": video, "output": str(output), "parameters": parameters, "device": device,
        "provenance": configuration["provenance"],
    }, legacy_environment(device)


def verify_sports_encoder(encoder, path):
    import torch

    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    state = checkpoint.get("state_dict", checkpoint)
    state = {k[7:] if k.startswith("module.") else k: v for k, v in state.items()}
    missing = [k for k, v in encoder.model.state_dict().items()
               if not k.startswith("classifier.") and (k not in state or not torch.equal(v.detach().cpu(), state[k]))]
    if missing:
        raise ValueError(f"Sports OSNet weights were not loaded on all inference layers: {missing[:10]}")


def gta_link(spec):
    import collections
    import numpy as np
    import torch
    from PIL import Image
    from torchvision import transforms as T

    from common import digest, json_write
    from evaluate import bbox, checked_tracks
    from jobs import capture

    if not torch.cuda.is_available():
        raise RuntimeError("GTA-Link feature extraction requires a functional CUDA device in the isolated environment")
    torch.cuda.set_device(0)
    torch.cuda.reset_peak_memory_stats(0)
    time_start = time.perf_counter()
    sys.path.insert(0, spec["repo"])
    from Tracklet import Tracklet
    from refine_tracklets import get_distance_matrix, get_spatial_constraints, merge_tracklets, split_tracklets
    from torchreid.utils import FeatureExtractor

    encoder = FeatureExtractor(model_name="osnet_x1_0", model_path=spec["checkpoint"], device="cuda:0")
    verify_sports_encoder(encoder, spec["checkpoint"])
    transform = T.Compose([T.Resize([256, 128]), T.ToTensor(),
                           T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    video = spec["video"]
    tracks = checked_tracks(spec["tracks"], video)
    inputs = {}
    original = collections.defaultdict(list)
    features_s = 0.0
    with capture(video["path"]) as source:
        for row in tracks:
            ok, frame = source.read()
            if not ok:
                raise RuntimeError("Original clip is shorter than its frozen track cache")
            if not row["tracks"]:
                continue
            pil = Image.fromarray(np.ascontiguousarray(frame[:, :, ::-1]))
            crops = [transform(pil.crop(tuple(x[coord] for coord in ("x1", "y1", "x2", "y2")))) for x in row["tracks"]]
            torch.cuda.synchronize()
            started = time.perf_counter()
            vectors = [encoder(torch.stack(crops[i:i + 32])).detach().cpu().numpy()
                       for i in range(0, len(crops), 32)]
            torch.cuda.synchronize()
            features_s += time.perf_counter() - started
            for item, vector in zip(row["tracks"], np.concatenate(vectors)):
                tid, frame_id = item["track_id"], row["frame"]
                xywh = bbox(item)
                track = inputs.setdefault(tid, Tracklet(track_id=tid))
                track.append_det(frame_id, item["score"], xywh)
                norm = np.linalg.norm(vector)
                if not np.isfinite(norm) or norm <= 0:
                    raise ValueError("Invalid original GTA-Link appearance feature")
                track.append_feat(vector / norm)
                original[(frame_id, tuple(xywh))].append(tid)
    if not inputs:
        raise Blocked("GTA-Link requires actual track observations")
    settings = spec["parameters"]
    spatial_x, spatial_y = get_spatial_constraints(inputs, settings["spatial_factor"])
    split = split_tracklets(inputs, eps=settings["eps"], min_samples=settings["min_samples"],
                            max_k=settings["max_k"], len_thres=settings["min_len"])
    split_count = len(split)
    merged = merge_tracklets(split, {}, get_distance_matrix(split), seq_name=video["clip"],
                             max_x_range=spatial_x, max_y_range=spatial_y,
                             merge_dist_thres=settings["merge_dist_thres"])
    mapped = collections.defaultdict(dict)
    for identity, track in sorted(merged.items()):
        for frame_id, xywh in zip(track.times, track.bboxes):
            parents = original[(frame_id, tuple(xywh))]
            if len(parents) != 1:
                raise ValueError("Cannot recover a unique input observation for an upstream GTA-Link box")
            tid = parents[0]
            if tid in mapped[frame_id]:
                raise ValueError("Upstream GTA-Link has a duplicate output for one input observation")
            mapped[frame_id][tid] = str(identity)
    outputs = sum(map(len, mapped.values()))
    json_write(spec["output"], {"method": "gta_link", "clip_sha256": video["sha256"],
                                "tracking_sha256": digest(spec["tracks"]), "frame_groups": mapped,
                                "source_commit": GTA_COMMIT, "checkpoint_sha256": GTA_WEIGHT_HASH,
                                "provenance": spec["provenance"], "parameters": settings, "offline": True,
                                "input_tracks": len(inputs), "split_tracks": split_count, "linked_tracks": len(merged),
                                "retained_observations": outputs, "dropped_observations": sum(map(len, original.values())) - outputs,
                                "feature_seconds": features_s, "wall_s": time.perf_counter() - time_start,
                                "peak_cuda_allocated_bytes": torch.cuda.max_memory_allocated(0),
                                "peak_cuda_reserved_bytes": torch.cuda.max_memory_reserved(0), "device": spec["device"]})


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("spec")
    config = json_read(parser.parse_args().spec)
    (gta_link if config["method"] == "gta_link" else parts)(config)
'''


In [ ]:
SOURCES['upstream_tracking.py'] = r'''"""Adapter for the author's Deep-EIoU association on externally frozen detections."""
import argparse
import time
from pathlib import Path
from types import SimpleNamespace

from backends import HUMANS
from common import BOX, Blocked, atomic_path, canonical_box, digest, freeze_predictions, json_read, json_write, write_line
from jobs import capture, overlay
from legacy import GTA_WEIGHT_HASH, legacy_environment, validate_checkout
from common import verified

COMMIT = "2fc7d270eec8290fda075b976b5e06683c788e1e"


def deep_profile(configuration, device, cache, video, output, thresholds, overlays):
    required = {"python", "repo", "sports_checkpoint", "rights_reviewed", "provenance"}
    if not required <= set(configuration) or configuration["rights_reviewed"] is not True or not configuration["provenance"]:
        raise Blocked("Deep-EIoU needs a rights-reviewed clean checkout, legacy Python, and the verified GTA-packaged sports OSNet weights.")
    python, repo = validate_checkout(configuration, COMMIT, "Deep-EIoU")
    checkpoint = verified(configuration["sports_checkpoint"], GTA_WEIGHT_HASH)
    settings = dict(track_high_thresh=thresholds["high"], track_low_thresh=thresholds["low"],
                    new_track_thresh=thresholds["new"], track_buffer=thresholds["buffer"], match_thresh=thresholds["match"],
                    with_reid=True, proximity_thresh=configuration.get("proximity_thresh", .5),
                    appearance_thresh=configuration.get("appearance_thresh", .25))
    return python, dict(source=str(repo / "Deep-EIoU"), checkpoint=str(checkpoint), cache=str(cache), video=video,
                        output=str(output), device=device, thresholds=settings, overlays=overlays,
                        provenance=configuration["provenance"]), legacy_environment(device)


def run(spec):
    import math
    import sys

    import cv2
    import numpy as np
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("Deep-EIoU appearance encoder requires CUDA in its isolated profile")
    if np.lib.NumpyVersion(np.__version__) >= "1.24.0":
        raise Blocked(f"The original tracker uses np.float; install NumPy <1.24 in the isolated profile (found {np.__version__}).")
    cv2.setNumThreads(1)
    torch.cuda.set_device(0)
    torch.cuda.reset_peak_memory_stats(0)
    started = time.perf_counter()
    sys.path.insert(0, spec["source"])
    from tracker.Deep_EIoU import Deep_EIoU
    from torchreid.utils import FeatureExtractor

    video = spec["video"]
    extractor = FeatureExtractor(model_name="osnet_x1_0", model_path=spec["checkpoint"], device="cuda:0")
    from legacy import verify_sports_encoder
    verify_sports_encoder(extractor, spec["checkpoint"])
    tracker = Deep_EIoU(SimpleNamespace(**spec["thresholds"]), frame_rate=max(1, round(video["nominal_fps"])))
    out = Path(spec["output"])
    tracks = out / "tracks" / f"{video['clip']}.jsonl"
    mot = tracks.with_suffix(".txt")
    annotated_path = out / "annotated_videos" / f"{video['clip']}.mp4"
    durations, count, boxes = 0., 0, 0
    with capture(video["path"]) as source, atomic_path(tracks) as temporary, temporary.open("w") as rows, \
            atomic_path(mot) as mot_tmp, mot_tmp.open("w") as mot_rows, overlay(annotated_path, video, spec["overlays"]) as annotated:
        for frame in freeze_predictions(spec["cache"], video):
            ok, image = source.read()
            if not ok:
                raise ValueError("Frozen detection cache outlasts the clip")
            people = [row for row in frame["detections"] if row["class"] in HUMANS]
            detections = np.asarray([[p[k] for k in (*BOX, "score")] for p in people], dtype=np.float32).reshape(-1, 5)
            crops = [image[max(0, math.floor(p["y1"])):math.ceil(p["y2"]),
                           max(0, math.floor(p["x1"])):math.ceil(p["x2"])] for p in people]
            torch.cuda.synchronize()
            ts = time.perf_counter()
            if crops:
                embs = np.concatenate([extractor(crops[i:i + 32]).detach().cpu().numpy() for i in range(0, len(crops), 32)])
            else:
                embs = np.empty((0, 512), dtype=np.float32)
            if embs.shape != (len(detections), 512) or not np.isfinite(embs).all():
                raise ValueError("Deep-EIoU expects one finite sports OSNet vector per frozen detection")
            predicted = tracker.update(detections, embs.copy())
            torch.cuda.synchronize()
            durations += time.perf_counter() - ts
            actual = []
            for current in predicted:
                if current.frame_id != frame["frame"]:
                    continue
                x, y, w, h = map(float, current.last_tlwh)
                vals = [x, y, x + w, y + h]
                matches = np.flatnonzero(np.max(np.abs(detections[:, :4] - vals), axis=1) < 1e-3)
                matches = [i for i in matches if abs(float(detections[i, 4]) - current.score) < 1e-7]
                if len(matches) != 1:
                    raise ValueError("Deep-EIoU observation cannot be mapped uniquely to a frozen detection")
                original = people[matches[0]]
                actual.append(dict(canonical_box(vals, (video["width"], video["height"])), track_id=int(current.track_id),
                                   score=float(current.score), detection_id=original["detection_id"], **{"class": original["class"]}))
                mot_rows.write(f"{frame['frame']},{current.track_id},{x+1:.3f},{y+1:.3f},{w:.3f},{h:.3f},"
                               f"{current.score:.7f},-1,-1,-1\n")
            count = frame["frame"]
            boxes += len(actual)
            write_line(rows, {"frame": count, "tracks": actual})
            annotated.draw(image, actual, True)
    json_write(tracks.with_suffix(".json"), {
        "clip": video["clip"], "clip_sha256": video["sha256"], "track_sha256": digest(tracks), "frames": count,
        "detection_sha256": digest(spec["cache"]), "method": "deep_eiou",
        "condition": "Original Deep-EIoU association with frozen detector and GTA-packaged sports OSNet x1.0",
        "source_commit": COMMIT, "checkpoint_sha256": GTA_WEIGHT_HASH, "provenance": spec["provenance"],
        "parameters": spec["thresholds"], "inference_s": durations, "wall_s": time.perf_counter() - started,
        "peak_cuda_allocated_bytes": torch.cuda.max_memory_allocated(0), "device": spec["device"], "boxes": boxes,
    })


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("spec")
    run(json_read(parser.parse_args().spec))
'''


In [ ]:
SOURCES['lab.py'] = r'''"""Standalone notebook coordinator. The notebook materializes this exact source."""
import os
import subprocess
import sys
import traceback
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

from advanced import ADVANCED, PART_COMMITS, compare_embeddings, homography, identity_output, identity_overlay, load_agentcoach, prepare_crops, temporal_ball, temporal_ocr
from backends import DETECTORS, TRACKERS, detector_asset, osnet_asset
from common import Blocked, GROUPS, NAMES, STATES, digest, export_zip, json_read, json_write, resolve_dataset, table, probe_endpoint
from environment import PACKAGE_PROFILES, command, device_plan, inventory, prepare_profile
from evaluate import (DETECTION_METRICS, IDENTITY_METRICS, REID_METRICS, TRACKING_METRICS,
                      identity_metrics, parse_reviewed)
from jobs import dataset_audit
from schedule import SAMPLE_FIELDS

STATUS_FIELDS = ("group", "method", "stage", "state", "reason", "clip", "device", "pid", "traceback")
RUNTIME_FIELDS = ("method", "phase", "clip", "device", "pid", "state", "frames", "crops_processed", "boxes", "clip_seconds",
                  "wall_s", "inference_s", "model_load_s", "cache_sha256", "prediction_sha256", "prediction_path",
                  "start_monotonic_s", "end_monotonic_s", "gpu_uuid", "inference_batches", "cuda_frames",
                  "cuda_peak_allocated_bytes", "cuda_peak_reserved_bytes", "cuda_allocated_bytes", "cuda_reserved_bytes")
DEVICE_FIELDS = ("group", "method", "phase", "device", "gpu_uuid", "pid", "frames", "cuda_frames",
                 "inference_batches", "inference_s", "successful_jobs", "failures")
EXIT_CODES = {"EXECUTED": 0, "READY": 0, "FAILED": 1, "BLOCKED": 2}
METRIC_FIELDS = ("group", "method", "metric", "state", "value", "reason", "gt_sha256")
MODES = {"PREFLIGHT", "SMOKE", "BENCHMARK", "ADVANCED", "REPORT"}


def show_failure(error):
    return ("BLOCKED" if isinstance(error, Blocked) else "FAILED",
            f"{type(error).__name__}: {error}", traceback.format_exc())


def validate(config):
    if config["mode"] not in MODES:
        raise ValueError(f"RUN_MODE must be one of {sorted(MODES)}")
    if not set(config["groups"]) <= GROUPS:
        raise ValueError(f"Unknown MODEL_GROUPS, available groups: {sorted(GROUPS)}")
    if config["mode"] in {"PREFLIGHT", "SMOKE", "BENCHMARK"} and not any(
            group in config["groups"] and config[key] for group, key in (("detection", "detectors"), ("tracking", "trackers"))):
        raise ValueError("Select at least one detector or tracker for preflight, smoke or benchmark.")
    if config["mode"] == "ADVANCED" and not config["advanced_methods"]:
        raise ValueError("ADVANCED requires at least one explicit ADVANCED_METHODS selection")
    if config["mode"] == "ADVANCED" and config["advanced_methods"] and not set(config["advanced_methods"]) <= set().union(*ADVANCED.values()):
        raise ValueError("Unknown ADVANCED_METHODS, see the method matrix in the notebook")
    if not set(config["detectors"]) <= set(DETECTORS) or not set(config["trackers"]) <= set(TRACKERS):
        raise ValueError(f"Unknown detector/tracker. Options: {list(DETECTORS)} / {list(TRACKERS)}")
    if any(len(config[key]) != len(set(config[key])) for key in ("detectors", "trackers", "advanced_methods")):
        raise ValueError("Method selections must be unique")
    if config["track_detector"] not in DETECTORS or config["identity_tracker"] not in {*TRACKERS, "deep_eiou"}:
        raise ValueError("Invalid TRACK_DETECTOR or IDENTITY_TRACKER")
    if config["precision"] not in {"fp32", "fp16"} or not 1 <= config["max_videos"] <= len(NAMES):
        raise ValueError("Use fp32/fp16 and MAX_VIDEOS between 1 and 12")
    if not 1 <= config["batch"] <= 32 or config["smoke_frames"] < 1 or config["timeout"] < 30:
        raise ValueError("Set batch 1–32, positive smoke frames and JOB_TIMEOUT_S >= 30")
    if not 0 <= config["score"] <= config["operating_score"] <= 1:
        raise ValueError("Require 0 <= detection score floor <= evaluation operating point <= 1")
    settings = config["tracker_settings"]
    if not 0 <= config["score"] <= settings["low"] < settings["high"] <= 1:
        raise ValueError("Detector floor must retain the low-score stage of ByteTrack and BoT-SORT")
    if not all(0 <= settings[key] <= 1 for key in ("new", "match")) or settings["buffer"] < 1:
        raise ValueError("Invalid tracker scores or buffer")
    if not 0 <= config["visibility_min"] <= 1 or not -1 <= config["identity_threshold"] <= 1:
        raise ValueError("Invalid visibility or identity threshold")
    if any(not -1 <= x <= 1 for x in config["identity_sweep"]):
        raise ValueError("Identity sweep thresholds must be within [-1, 1]")
    if min(config["crop_stride"], config["crops_per_track"], config["distance_sample"]) < 1:
        raise ValueError("Crop limits and sample size must be positive")
    if not set(config["scaling_models"]) <= set(DETECTORS):
        raise ValueError("Unknown detector in SCALING_MODELS")


def new_run(root):
    name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
    path = root / "runs" / name
    path.mkdir(parents=True)
    return path


def find_run(root, name):
    if not name:
        raise Blocked("Supply CACHE_RUN with a saved run ID or extracted result directory")
    path = Path(name)
    path = path if path.is_absolute() else root / "runs" / path
    if not (path / "configuration.json").is_file():
        raise Blocked(f"No saved Scoutra run at {path}")
    return path


def groups_for(method):
    for group, entries in ADVANCED.items():
        if method in entries:
            return group
    raise ValueError(method)


class Laboratory:
    def __init__(self, root, config, run):
        self.root, self.config, self.run = root, config, run
        self.workspace = root / ".runtime"
        self.assets = self.workspace / "assets"
        self.network = {}
        self.readiness = {}
        self.detector_assets = {}
        self.cuda_profiles = {}
        self.status, self.runtime, self.metrics, self.samples = [], [], [], []
        self.schedules, self.comparisons, self.device_rows = [], [], []
        self.profiles, self.video_by_clip, self.selected = {}, {}, []
        self.devices, self.device_failure = None, None
        self.cache_run = None
        self.generated_crops = None
        environment = run / "environment"
        environment.mkdir()
        (environment / "installation_log.txt").write_text("Package installation occurs only for selected, enabled methods.\n")
        (environment / "dependencies.txt").write_text(f"Host Python {sys.version}\n")
        json_write(run / "configuration.json", config)

    def log(self, group, method, stage, state, reason="", *, brief=True, **details):
        if state not in STATES:
            raise ValueError(f"Invalid status {state}")
        self.status.append(dict(group=group, method=method, stage=stage, state=state, reason=reason, **details))
        if brief:
            print(f"{state:13} {group}/{method} [{stage}] {reason}", flush=True)

    def record_error(self, group, method, stage, error):
        state, reason, trace = show_failure(error)
        self.log(group, method, stage, state, reason, traceback=trace)
        return state, reason

    def hardware(self):
        if self.devices is not None:
            return self.devices
        if self.device_failure:
            raise Blocked(self.device_failure)
        try:
            self.devices = device_plan(self.gpu, self.config["use_both"], self.config["allow_single"])
        except Blocked as error:
            self.device_failure = str(error)
            raise
        print(f"Assigned logical devices: {self.devices}", flush=True)
        if len(self.devices) < 2:
            self.log("environment", "gpu", "device_plan", "EXECUTED", f"Explicit single-GPU mode on {self.devices[0]}")
        elif any("T4" not in x.get("name", "") for x in self.gpu["gpus"] if f"cuda:{x['index']}" in self.devices):
            self.log("environment", "gpu", "device_plan", "EXECUTED", "Two functioning CUDA devices; at least one is a different model from Tesla T4.")
        else:
            self.log("environment", "gpu", "device_plan", "EXECUTED", "Two independently validated T4 devices.")
        return self.devices

    def profile(self, name):
        if name not in self.profiles:
            try:
                self.profiles[name] = prepare_profile(name, self.workspace, self.run, self.config["auto_acquire"],
                                                      self.config.get("wheelhouse"), self.probe)
                self.log("environment", name, "profile", "EXECUTED", "Isolated import test and pip check passed")
            except Exception as error:
                self.profiles[name] = error
                self.record_error("environment", name, "profile", error)
        result = self.profiles[name]
        if isinstance(result, Exception):
            raise result
        return result

    def probe(self, url):
        if url not in self.network:
            try:
                self.network[url] = probe_endpoint(url)
            except OSError as error:
                self.network[url] = {"reachable": False, "error": f"{type(error).__name__}: {error}"}
        result = self.network[url]
        if not result["reachable"]:
            raise Blocked(f"Endpoint unreachable: {url}. {result['error']}. Enable Kaggle Internet or use local assets and WHEELHOUSE.")

    def environment_preflight(self):
        self.gpu = inventory(self.run / "environment" / "gpu_inventory.json")
        self.log("environment", "gpu", "inventory", "EXECUTED",
                 f"{self.gpu['gpu_count']} CUDA device(s) observed. See kernel test results.")
        json_write(self.run / "environment" / "profiles.json", PACKAGE_PROFILES)
        if self.config["cache_run"]:
            self.cache_run = find_run(self.root, self.config["cache_run"])

    def audit_dataset(self, root):
        records, errors = dataset_audit(root)
        if records:
            table(self.run / "dataset" / "manifest.csv", records, tuple(records[0]))
        for e in errors:
            self.log("dataset", "video_decode", "audit", "FAILED", e["error"], clip=e["file"])
        if errors:
            raise Blocked(f"{len(errors)} of 12 exact clips failed validation. The dataset manifest contains only verified clips.")
        self.videos = records
        self.video_by_clip = {video["clip"]: video for video in records}
        self.selected = records[:self.config["max_videos"]]
        self.log("dataset", "zhanto_highlights", "audit", "EXECUTED",
                 f"{len(records)} of 12 clips decoded and hashed. {len(self.selected)} selected for this run.")

    def cuda_profile(self, name):
        python, env = self.profile(name)
        if name not in self.cuda_profiles:
            expected = {p["index"]: p["uuid"] for p in self.gpu["gpus"] if p["functional"]}
            probe = inventory(self.run / "environment" / f"{name}_gpu_inventory.json", python, env)
            actual = {p["index"]: p["uuid"] for p in probe["gpus"] if p["functional"]}
            missing = [device for device in self.hardware()
                       if (i := int(device.split(":")[1])) not in actual or actual[i] != expected[i]]
            self.cuda_profiles[name] = missing
        missing = self.cuda_profiles[name]
        if missing:
            raise RuntimeError(f"Isolated profile {name} failed CUDA/UUID preflight for {missing}; see its GPU inventory.")

    def preflight_check(self, group, method, stage, action):
        try:
            result = action()
        except Exception as error:
            self.record_error(group, method, stage, error)
            return False, None
        self.log(group, method, stage, "EXECUTED")
        return True, result

    def preflight(self):
        config = self.config
        print(f"Preflight on Python {sys.version.split()[0]} | "
              f"acquisition={'on' if config['auto_acquire'] else 'off'} | "
              f"asset cache={config.get('asset_cache') or 'session cache'} | "
              f"wheelhouse={config.get('wheelhouse') or 'none'}", flush=True)

        def check_encoder():
            from shutil import which

            if not which("ffmpeg"):
                raise Blocked("Annotated videos require ffmpeg. Install it or set SAVE_ANNOTATED_VIDEOS=False.")
            result = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True, timeout=15)
            if result.returncode or " libx264 " not in result.stdout:
                raise Blocked("Annotated videos require the ffmpeg libx264 encoder.")
        encoder_ok, _ = (self.preflight_check("environment", "ffmpeg", "encoder_preflight", check_encoder)
                         if config["save_videos"] else (True, None))
        if "detection" in config["groups"]:
            for method in config["detectors"]:
                gpu_ok, _ = self.preflight_check("detection", method, "gpu_preflight", self.hardware)
                if not gpu_ok or not encoder_ok:
                    continue
                deps_ok, _ = self.preflight_check("detection", method, "dependency_preflight",
                                                  lambda: self.cuda_profile(DETECTORS[method]))
                assets_ok, asset = self.preflight_check("detection", method, "asset_preflight",
                                                        lambda: detector_asset(method, config, self.assets, self.probe))
                self.readiness[("detection", method)] = deps_ok and assets_ok
                if assets_ok:
                    self.detector_assets[method] = asset
                    json_write(self.run / "environment" / "assets" / f"{method}.json", asset)
        if "tracking" in config["groups"]:
            for method in config["trackers"]:
                if not encoder_ok:
                    continue
                if self.cache_run is None and not self.readiness.get(("detection", config["track_detector"])):
                    self.log("tracking", method, "cache_preflight", "BLOCKED",
                             f"Selected detector {config['track_detector']} is unavailable and CACHE_RUN is unset.")
                    continue
                if method == "strongsort":
                    gpu_ok, _ = self.preflight_check("tracking", method, "gpu_preflight", self.hardware)
                    asset_ok, _ = self.preflight_check("tracking", method, "asset_preflight", lambda: osnet_asset(config))
                    if not (gpu_ok and asset_ok):
                        continue
                setup = self.cuda_profile if method == "strongsort" else self.profile
                ready, _ = self.preflight_check("tracking", method, "dependency_preflight",
                                                 lambda: setup(TRACKERS[method]))
                self.readiness[("tracking", method)] = ready

    def schedule(self, group, method, phase, tasks, devices, profile):
        python, env = self.profile(profile)
        logs = self.run / "environment" / "workers" / group / method / phase
        logs.mkdir(parents=True, exist_ok=True)
        request = logs / "schedule_input.json"
        response = logs / "schedule_output.json"
        active = devices[:len(tasks)]
        telemetry = any(device.startswith("cuda") for device in active)
        gpu_uuids = {f"cuda:{gpu['index']}": gpu["uuid"] for gpu in self.gpu["gpus"]
                     if gpu.get("functional") and gpu.get("uuid")}
        json_write(request, {"tasks": tasks, "devices": active, "logdir": str(logs), "device_uuids": gpu_uuids,
                             "timeout": self.config["timeout"], "telemetry": telemetry})
        with (logs / "scheduler.log").open("w") as log:
            proc = subprocess.Popen([python, "-u", str(Path(__file__).with_name("schedule.py")), str(request), str(response)],
                                    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
            try:
                for line in proc.stdout:
                    log.write(line)
                    print(line.rstrip(), flush=True)
                code = proc.wait()
            finally:
                if proc.poll() is None:
                    import signal
                    os.killpg(proc.pid, signal.SIGTERM)
                    proc.wait()
            if code or not response.is_file():
                raise RuntimeError(f"Scheduler failed with exit code {code}. See {logs / 'scheduler.log'}.")
        report = json_read(response)
        rows = report["results"]
        for row in rows:
            if row["state"] == "EXECUTED":
                reason = f"{row.get('frames', row.get('crops_processed', 0))} items processed"
            else:
                reason = row["reason"]
            self.log(group, method, phase, row["state"], reason, brief=row["state"] != "EXECUTED",
                     clip=row["clip"], device=row.get("device", ""), pid=row.get("pid", ""), traceback=row.get("traceback", ""))
        successes = [row for row in rows if row["state"] == "EXECUTED"]
        total_frames = sum(row.get("frames", 0) for row in successes)
        wall = report["wall_s"]
        used = sorted({row["device"] for row in successes})
        confirmed = True
        if group == "detection" and telemetry:
            confirmed = validate_detector_devices(successes, active, gpu_uuids)
            if not confirmed:
                self.log(group, method, f"{phase}_cuda_inference", "FAILED",
                         "Expected CUDA frames and inference batches on every assigned physical GPU, with a distinct UUID and PID per worker.")
        self.device_rows.extend({**row, "group": group, "method": method, "phase": phase} for row in report["workers"])
        performance = {
            "group": group, "method": method, "phase": phase, "planned_devices": active, "working_devices": used,
            "cuda_inference_verified": confirmed if group == "detection" and telemetry else None,
            "jobs_succeeded": len(successes), "jobs_expected": len(tasks), "frames": total_frames,
            "wall_s": wall, "frames_per_s": total_frames / wall,
            "video_seconds_per_wall_s": sum(row.get("clip_seconds", 0) for row in successes) / wall,
            "device_wall_hours": wall * len(active) / 3600 if telemetry else 0,
            "samples": len(report["telemetry_samples"]), "telemetry_diagnostics": report["telemetry_diagnostics"],
            "workers": report["workers"],
        }
        if len(used) < len(active):
            self.log(group, method, f"{phase}_coverage", "FAILED", f"Completed work on {used}; expected {active}. Inspect worker failures.")
        print(f"{method} {phase}: {len(successes)}/{len(tasks)} tasks, {total_frames} frames, "
              f"{wall:.2f}s, devices {used}", flush=True)
        self.runtime.extend(rows)
        self.schedules.append(performance)
        self.samples.extend({**row, "method": method, "phase": phase} for row in report["telemetry_samples"])
        return successes, performance

    def detection_tasks(self, method, asset, phase, videos, limit, overlays):
        return [{"id": v["clip"], "method": method, "phase": phase, "kind": "detect", "video": v,
                 "cost": min(v["frames"], limit or v["frames"]), "limit": limit, "batch": self.config["batch"],
                 "asset": asset, "score": self.config["score"], "precision": self.config["precision"], "overlays": overlays,
                 "output": str(self.run / "detections" / method / phase)} for v in videos]

    def detect(self, method):
        devices = self.hardware()
        asset = self.detector_assets[method]
        smoke_count = len(devices)
        videos = sorted(self.selected, key=lambda row: -row["frames"])[:smoke_count]
        profile = DETECTORS[method]
        tasks = self.detection_tasks(method, asset, "smoke", videos, self.config["smoke_frames"], self.config["save_videos"])
        smoke, smoke_info = self.schedule("detection", method, "smoke", tasks, devices, profile)
        if self.config["mode"] == "SMOKE":
            return
        if len(smoke) != len(tasks) or smoke_info["cuda_inference_verified"] is False:
            self.log("detection", method, "benchmark", "BLOCKED", "The real model smoke test failed on at least one assigned device.")
            return
        print(f"{method}: smoke passed, starting full inference over {len(self.selected)} clips.", flush=True)
        tasks = self.detection_tasks(method, asset, "benchmark", self.selected, None, self.config["save_videos"])
        predictions, dual = self.schedule("detection", method, "benchmark", tasks, devices, profile)
        self.evaluate("detection", method, predictions)
        if method in self.config["scaling_models"] and len(devices) == 2 and len(tasks) >= 2:
            if len(predictions) != len(tasks) or dual["cuda_inference_verified"] is False:
                self.log("detection", method, "scaling", "BLOCKED", "Two-GPU workload did not fully complete.")
                return
            trial = self.detection_tasks(method, asset, "scaling_single", self.selected, None, self.config["save_videos"])
            results, single = self.schedule("detection", method, "scaling_single", trial, devices[:1], profile)
            frames_dual = {row["clip"]: row["frames"] for row in predictions}
            frames_single = {row["clip"]: row["frames"] for row in results}
            if frames_dual != frames_single or single["cuda_inference_verified"] is False:
                self.log("detection", method, "scaling", "NOT_EVALUATED", "Single- and dual-GPU runs processed different workloads.")
                return
            speedup = single["wall_s"] / dual["wall_s"]
            inputs = {row["clip"]: row["sha256"] for row in self.selected}
            self.comparisons.append({"method": method, "clips": len(inputs), "frames": dual["frames"],
                                     "dual_wall_s": dual["wall_s"], "single_wall_s": single["wall_s"],
                                     "speedup_single_over_dual": speedup, "clip_hashes": inputs,
                                     "order": "dual then single", "devices": devices, "batch": self.config["batch"],
                                     "precision": self.config["precision"], "overlays": self.config["save_videos"]})
            self.log("detection", method, "scaling", "EXECUTED", f"Measured speedup {speedup:.3f}× on identical inputs.")
        elif method in self.config["scaling_models"]:
            self.log("detection", method, "scaling", "NOT_EVALUATED", "A scaling measurement needs two GPUs and at least two clips.")

    def cached(self, kind, method, clip, phase="benchmark"):
        video = self.video_by_clip[clip]
        directory = "predictions" if kind == "detections" else "tracks"
        for source in (self.run, self.cache_run):
            if source:
                path = source / kind / method / phase / directory / f"{clip}.jsonl"
                if not path.exists():
                    continue
                meta = json_read(path.with_suffix(".json"))
                key = "prediction_sha256" if kind == "detections" else "track_sha256"
                if meta["clip_sha256"] != video["sha256"] or digest(path) != meta[key]:
                    raise ValueError(f"Cached artifacts fail input/output fingerprint validation: {path}")
                if phase == "benchmark" and meta["frames"] != video["frames"]:
                    raise ValueError(f"Saved benchmark incomplete: {path}")
                return path
        raise Blocked(f"Missing {phase} cache for {kind}/{method}/{clip}. Run the source method or set CACHE_RUN.")

    def track(self, method):
        devices = self.hardware() if method == "strongsort" else ["cpu:0", "cpu:1"]
        asset = osnet_asset(self.config) if method == "strongsort" else None
        phase = "smoke" if self.config["mode"] == "SMOKE" else "benchmark"
        videos = (sorted(self.selected, key=lambda item: -item["frames"])[:(len(self.devices) if self.devices else 1)]
                  if phase == "smoke" else self.selected)
        tasks = []
        for video in videos:
            try:
                raw = self.cached("detections", self.config["track_detector"], video["clip"], phase)
                if json_read(raw.with_suffix(".json"))["score_floor"] > self.config["tracker_settings"]["low"]:
                    raise ValueError("Selected detector cache removed scores needed by the low-score tracking stage")
                tasks.append({"id": video["clip"], "method": method, "phase": phase, "kind": "track", "video": video,
                              "cost": video["frames"], "cache": str(raw), "thresholds": self.config["tracker_settings"],
                              "asset": asset, "overlays": self.config["save_videos"],
                              "output": str(self.run / "tracking" / method / phase)})
            except Exception as error:
                state, reason, trace = show_failure(error)
                self.log("tracking", method, phase, state, reason, clip=video["clip"], traceback=trace)
        if not tasks:
            return
        rows, _ = self.schedule("tracking", method, phase, tasks, devices, TRACKERS[method])
        if phase == "benchmark":
            self.evaluate("tracking", method, rows)

    def capture_metrics(self, group, method, names, values=None, reason="", fingerprint=""):
        for name in names:
            result = (values or {}).get(name)
            state = "EXECUTED" if result is not None else "NOT_EVALUATED"
            self.metrics.append(dict(group=group, method=method, metric=name, state=state,
                                     reason=("" if result is not None else reason or "No eligible annotated positives or denominator"),
                                     value=result, gt_sha256=fingerprint))
        if any(row["group"] == group and row["method"] == method and row["state"] == "EXECUTED" for row in self.metrics):
            self.log(group, method, "evaluation", "EXECUTED", "Reviewed annotations evaluated; see metrics.csv.")
        else:
            self.log(group, method, "evaluation", "NOT_EVALUATED", reason or "No metric has an eligible labeled denominator.")

    def evaluate(self, group, method, predictions):
        names = DETECTION_METRICS if group == "detection" else TRACKING_METRICS
        key = "detection_annotations" if group == "detection" else "tracking_annotations"
        gt = self.config[key]
        if not predictions or not gt:
            reason = "No successful full-clip predictions" if not predictions else "No human-reviewed annotation file selected."
            self.capture_metrics(group, method, names, reason=reason)
            return
        try:
            python, env = self.profile("eval")
            folder = self.run / "benchmarks" / "evaluation" / group / method
            spec, result = folder / "input.json", folder / "metrics.json"
            data = dict(kind=group, annotations=gt, predictions={p["clip"]: p["prediction_path"] for p in predictions},
                        inputs={v["clip"]: v for v in self.selected}, operating_point=self.config["operating_score"], cache_floor=self.config["score"])
            json_write(spec, data)
            code = command([python, str(Path(__file__).with_name("evaluate.py")), str(spec), str(result)],
                           folder / "calculation.log", environment=env)
            if code:
                raise RuntimeError(f"Evaluation exit={code}; see {folder / 'calculation.log'}")
            output = json_read(result)
            self.capture_metrics(group, method, names, {name: v["value"] for name, v in output["metrics"].items()},
                                 fingerprint=output["gt_sha256"])
        except Exception as error:
            _, reason = self.record_error(group, method, "evaluation_error", error)
            self.capture_metrics(group, method, names, reason=reason)

    def reid(self, method):
        if method == "prepare_crops":
            output = self.run / "reid" / "crops" / "manifest.csv"
            count = prepare_crops(lambda clip: self.cached("tracking", self.config["identity_tracker"], clip),
                                  self.selected, output, self.config["crop_stride"], self.config["crops_per_track"])
            self.generated_crops = str(output)
            self.log("reid", method, "advanced", "EXECUTED", f"{count} immutable crops stored")
            return
        crops = self.config["crop_manifest"] or self.generated_crops
        if not crops:
            raise Blocked("Set CROP_MANIFEST to a set of frozen image crops with crop_id,path,sha256. See annotation guide.")
        output = self.run / "reid" / "embeddings" / f"{method}.npz"
        if method == "osnet":
            devices = self.hardware()
            asset = osnet_asset(self.config)
            task = dict(id="crop_manifest", method=method, phase="advanced", kind="osnet", batch=self.config["batch"],
                        crops=crops, asset=asset, output=str(output))
            rows, _ = self.schedule("reid", method, "advanced", [task], devices[:1], "boxmot")
            if not rows:
                return
        else:
            self.part_reid(method, crops, output)
        comparisons = self.run / "reid" / "comparisons" / f"{method}.json"
        reviewed = self.config["reid_annotations"]
        result = compare_embeddings(comparisons, output, reviewed, self.config["distance_sample"],
                                    self.config["visibility_min"], self.config["reid_threshold"])
        self.capture_metrics("reid", method, REID_METRICS, result.get("metrics"), fingerprint=result.get("gt_sha256", ""),
                             reason="" if reviewed else "No human-reviewed query/gallery split selected.")

    def part_reid(self, method, crops, output):
        from legacy import part_profile
        python, spec, env = part_profile(method, self.config["legacy_reid"], self.hardware()[0], crops, output, self.config["batch"])
        request = self.run / "reid" / f"{method}_legacy_input.json"
        json_write(request, spec)
        folder = self.run / "environment"
        logfile = folder / f"{method}_legacy.log"
        if command([python, str(Path(__file__).with_name("legacy.py")), str(request)], logfile,
                   timeout=self.config["timeout"], environment=env) or not output.is_file():
            raise RuntimeError(f"Isolated upstream extraction failed. See {logfile}.")
        self.log("reid", method, "advanced", "EXECUTED", "Full upstream feature extraction completed. See embedding sidecar.")

    def advanced(self, group, method):
        blocker = ADVANCED[group][method]
        if blocker:
            raise Blocked(blocker)
        if group == "reid":
            self.reid(method)
            return
        if method == "deep_eiou":
            self.deep_eiou()
            return
        if method == "temporal_ocr_vote":
            source = self.config["ocr_predictions"]
            if not source:
                raise Blocked("Set OCR_PREDICTIONS to saved frame-level digit predictions.")
            count = temporal_ocr(source, self.run / "jersey" / "temporal_ocr_vote.json")
            self.log(group, method, "advanced", "EXECUTED", f"{count} tracklets aggregated")
            return
        if group == "calibration" and not self.config["landmarks"]:
            raise Blocked("Set PITCH_LANDMARKS with reviewed framewise pixel-to-meter correspondences.")
        module = load_agentcoach(self.assets, self.config["auto_acquire"]) if method == "agentcoach_reid_only" else None
        for video in self.selected:
            clip = video["clip"]
            try:
                if group == "identity":
                    raw = self.cached("tracking", self.config["identity_tracker"], clip)
                    output = self.run / "identity" / "results" / method / f"{clip}.json"
                    if method == "gta_link":
                        from legacy import gta_profile

                        python, request, env = gta_profile(self.config["gta_config"], self.hardware()[0], raw, video, output)
                        spec = self.run / "identity" / "legacy" / f"{clip}.json"
                        logfile = self.run / "environment" / f"gta_{clip}.log"
                        json_write(spec, request)
                        if command([python, str(Path(__file__).with_name("legacy.py")), str(spec)], logfile,
                                   timeout=self.config["timeout"], environment=env) or not output.is_file():
                            raise RuntimeError(f"Original GTA-Link runner failed. See {logfile}.")
                        count = json_read(output)["retained_observations"]
                    else:
                        count = identity_output(method, raw, video, output, self.config["identity_features"],
                                                module, self.config["identity_threshold"] if method != "no_link" else None)
                    if self.config["save_videos"]:
                        identity_overlay(raw, output, video, self.run / "identity" / "visualizations" / method / f"{clip}.mp4")
                elif group == "calibration":
                    output = self.run / group / method / f"{clip}.json"
                    count = homography(video, self.config["landmarks"], output)
                else:
                    raw = self.cached("detections", self.config["track_detector"], clip)
                    output = self.run / "ball" / method / f"{clip}.json"
                    count = temporal_ball(raw, video, output, self.config["operating_score"], .06, 10)
                self.log(group, method, "advanced", "EXECUTED", f"{count} items processed", clip=clip)
            except Exception as error:
                state, reason, trace = show_failure(error)
                self.log(group, method, "advanced", state, reason, clip=clip, traceback=trace)
        if group == "identity":
            self.evaluate_identity(method)
            if method == "cosine_tracklet_link":
                for threshold in self.config["identity_sweep"]:
                    scenario = f"{method}@{threshold:g}"
                    for video in self.selected:
                        clip = video["clip"]
                        try:
                            raw = self.cached("tracking", self.config["identity_tracker"], clip)
                            output = self.run / "identity" / "results" / scenario / f"{clip}.json"
                            count = identity_output(method, raw, video, output, self.config["identity_features"], threshold=threshold)
                            self.log(group, scenario, "advanced", "EXECUTED", f"{count} local tracklets", clip=clip)
                        except Exception as error:
                            state, reason, trace = show_failure(error)
                            self.log(group, scenario, "advanced", state, reason, clip=clip, traceback=trace)
                    self.evaluate_identity(scenario)

    def deep_eiou(self):
        from upstream_tracking import deep_profile

        name = "deep_eiou"
        if not self.config["deep_eiou_config"]:
            raise Blocked("Set DEEP_EIOU_CONFIG with the rights-reviewed source, isolated NumPy <1.24 environment and verified sports OSNet weights.")
        device = self.hardware()[0]
        predictions = []
        for video in self.selected:
            clip = video["clip"]
            try:
                cache = self.cached("detections", self.config["track_detector"], clip)
                output = self.run / "tracking" / name / "benchmark"
                python, spec, env = deep_profile(self.config["deep_eiou_config"], device, cache, video, output,
                                                 self.config["tracker_settings"], self.config["save_videos"])
                request = self.run / "environment" / f"{name}_{clip}.json"
                logfile = request.with_suffix(".log")
                json_write(request, spec)
                tracker = Path(__file__).with_name("upstream_tracking.py")
                code = command([python, str(tracker), str(request)], logfile, timeout=self.config["timeout"], environment=env)
                result = output / "tracks" / f"{clip}.jsonl"
                if code or not result.is_file():
                    raise RuntimeError(f"Isolated Deep-EIoU process failed. See {logfile}.")
                info = json_read(result.with_suffix(".json"))
                self.log("tracking", name, "advanced", "EXECUTED", f"{info['frames']} frames", clip=clip, device=device)
                row = dict(method=name, phase="advanced", clip=clip, state="EXECUTED", device=device,
                           frames=info["frames"], boxes=info["boxes"], wall_s=info["wall_s"], inference_s=info["inference_s"],
                           cuda_peak_allocated_bytes=info["peak_cuda_allocated_bytes"],
                           prediction_path=str(result), cache_sha256=info["detection_sha256"])
                self.runtime.append(row)
                predictions.append(row)
            except Exception as error:
                state, reason, trace = show_failure(error)
                self.log("tracking", name, "advanced", state, reason, clip=clip, traceback=trace)
        self.evaluate("tracking", name, predictions)

    def evaluate_identity(self, method):
        spec = self.config["identity_annotations"]
        if not spec:
            self.capture_metrics("identity", method, IDENTITY_METRICS,
                                 reason="No human-reviewed continuous scenes, anchors or re-entry episodes selected.")
            return
        per_clip = {}
        try:
            reviewed = parse_reviewed(spec, self.video_by_clip, dense=True)
            annotations = json_read(spec)["clips"]
            for clip, item in reviewed.items():
                output = self.run / "identity" / "results" / method / f"{clip}.json"
                if not output.is_file():
                    continue
                mapped = json_read(output)
                raw = self.cached("tracking", self.config["identity_tracker"], clip)
                per_clip[clip] = identity_metrics(raw, mapped, self.video_by_clip[clip], annotations[clip], item)
            if not per_clip:
                raise Blocked("No reviewed clip overlaps with successful identity output")
            stats = Counter()
            for x in per_clip.values():
                stats.update(x["counts"])
            delays = sorted(delay for x in per_clip.values() for delay in x["recovery_delays_frames"])

            def fraction(key, denom):
                return stats[key] / stats[denom] if stats[denom] else None

            totals = dict(reentry_accuracy=fraction("recoveries", "return_episodes"),
                          identity_precision=fraction("correct_person_frames", "assigned_person_frames"),
                          identity_coverage=fraction("assigned_person_frames", "gt_person_frames"),
                          recovery_delay_frames=((delays[(len(delays) - 1) // 2] + delays[len(delays) // 2]) / 2
                                                 if delays else None))
            totals.update({key: sum(x["metrics"][key] or 0 for x in per_clip.values()) if stats["gt_person_frames"] else None
                           for key in ("false_merges", "false_splits")})
            fingerprint = digest(spec)
            json_write(self.run / "identity" / "evaluations" / f"{method}.json",
                       {"gt_sha256": fingerprint, "metrics": totals, "per_clip": per_clip})
            self.capture_metrics("identity", method, IDENTITY_METRICS, totals, fingerprint=fingerprint)
        except Exception as error:
            _, reason = self.record_error("identity", method, "evaluation_error", error)
            self.capture_metrics("identity", method, IDENTITY_METRICS, reason=reason)

    def record_unrun(self):
        advertised = {group: list(methods) for group, methods in ADVANCED.items()}
        advertised["detection"][:0] = DETECTORS
        advertised["tracking"][:0] = TRACKERS
        for group, methods in advertised.items():
            for method in methods:
                if not any(r["group"] == group and r["method"] == method and not r["stage"].endswith("_preflight") for r in self.status):
                    if self.config["mode"] == "PREFLIGHT":
                        reason = "PREFLIGHT does not run inference."
                    elif (group, method) in self.readiness or any(
                            r["group"] == group and r["method"] == method for r in self.status):
                        reason = "Preflight finished; no inference job was completed for this method."
                    else:
                        reason = "Method not selected in this run."
                    blocker = ADVANCED[group].get(method)
                    if blocker:
                        reason = f"{reason} {blocker}"
                    self.log(group, method, "execution", "SKIPPED", reason, brief=False)
                names = {"detection": DETECTION_METRICS, "tracking": TRACKING_METRICS,
                         "reid": REID_METRICS, "identity": IDENTITY_METRICS}.get(group)
                if names and not any(r["group"] == group and r["method"] == method for r in self.metrics):
                    reason = "No eligible reviewed benchmark executed."
                    self.metrics.extend(dict(group=group, method=method, metric=n, state="NOT_EVALUATED", reason=reason) for n in names)

    def core(self, group, methods, handler):
        if group not in self.config["groups"]:
            return
        for method in methods:
            if not self.readiness.get((group, method)):
                continue
            try:
                handler(method)
            except Exception as error:
                self.record_error(group, method, "execution", error)

    def reports(self):
        self.record_unrun()
        result = self.result()
        json_write(self.run / "RUN_RESULT.json", result)
        json_write(self.run / "environment" / "network.json",
                   {"auto_acquire": self.config["auto_acquire"], "wheelhouse": self.config.get("wheelhouse"),
                    "probes": self.network, "note": "No HTTP probe is needed for an already validated local profile or checkpoint."})
        bench = self.run / "benchmarks"
        table(bench / "experiment_status.csv", self.status, STATUS_FIELDS)
        table(bench / "metrics.csv", self.metrics, METRIC_FIELDS)
        if self.runtime:
            table(bench / "runtime.csv", self.runtime, RUNTIME_FIELDS)
        if self.device_rows:
            table(bench / "device_frames.csv", self.device_rows, DEVICE_FIELDS)
        if self.samples:
            table(bench / "gpu_usage.csv", self.samples, ("method", "phase", *SAMPLE_FIELDS))
        json_write(bench / "execution_summary.json", {"schedules": self.schedules, "scaling": self.comparisons})
        report = self.run / "reports"
        report.mkdir(exist_ok=True)
        counts = Counter(row["state"] for row in self.status)
        (report / "summary.md").write_text(
            f"# Scoutra run {self.run.name}\n\n**{result['state']}** · {self.config['mode']} · "
            f"{result['detection_frames']} detector frames, {result['cuda_detection_frames']} on CUDA\n\n"
            + "\n".join(f"- {name}: {count}" for name, count in sorted(counts.items()))
            + "\n\nSee benchmarks/experiment_status.csv for each method and clip. A hardware or asset check is distinct from inference.\n"
            + ("\nGPU samples are device-wide nvidia-smi utilization. Match by UUID in environment/gpu_inventory.json.\n" if self.samples else
               "\nNo device-wide utilization samples were recorded. See execution_summary.json for the sampler diagnostics and run scope.\n")
            + "\nInference time includes model preprocessing/postprocessing, with CUDA synchronization for GPU workers. "
              "Wall time includes cold worker/model startup, decode, inference, serialization and enabled overlays. "
              "Device-wall hours use planned device count; they are a scheduling cost estimate.\n", encoding="utf-8")
        failures = [row for row in self.status if row["state"] in {"FAILED", "BLOCKED"}]
        details = [f"## {row['group']}/{row['method']} · {row['stage']} · {row.get('clip', '')}\n\n"
                   f"{row['state']}: {row['reason']}" + (f"\n\n```text\n{row['traceback']}```" if row.get("traceback") else "")
                   for row in failures]
        (report / "failures.md").write_text("# Failures and blockers\n\n" + ("\n\n".join(details) if details else
                                             "No failed or blocked stages recorded. See experiment_status.csv for execution scope.") + "\n")
        mapping = ["# Method coverage", "", "| Group | Method | Stage | States |", "|---|---|---|---|"]
        by_key = {}
        for row in self.status:
            key = (row["group"], row["method"], row["stage"])
            by_key.setdefault(key, Counter())[row["state"]] += 1
        for key, counts in sorted(by_key.items()):
            state = ", ".join(f"{n} {s}" for s, n in sorted(counts.items()))
            mapping.append(f"| {' | '.join((*key, state))} |")
        mapping.extend(["", "## Same-workload GPU scaling", ""])
        mapping.extend([f"- {x['method']}: {x['frames']} frames, single {x['single_wall_s']:.2f}s, dual {x['dual_wall_s']:.2f}s, "
                        f"speedup {x['speedup_single_over_dual']:.3f}×." for x in self.comparisons] or ["No complete scaling measurement."])
        mapping.extend(["", "Tracking uses one frozen detector and clip-local IDs. BoT-SORT uses camera compensation and has appearance disabled.",
                        "Inspect cache fingerprints before comparing runs. Accuracy metrics use only reviewed labels.",
                        "The dataset consists of independent short highlights. Full-match identity reliability is outside this measurement."])
        (report / "method_comparison.md").write_text("\n".join(mapping) + "\n", encoding="utf-8")

    def result(self):
        states = {r["state"] for r in self.status}
        mode = self.config["mode"]
        detection_rows = [r for r in self.runtime if r["method"] in DETECTORS and r["state"] == "EXECUTED"]
        if "FAILED" in states:
            state = "FAILED"
        elif "BLOCKED" in states:
            state = "BLOCKED"
        elif mode == "PREFLIGHT":
            state = "READY"
        elif any(r["group"] in GROUPS and r["state"] == "EXECUTED"
                 and r["stage"] in {"smoke", "benchmark", "advanced"} for r in self.status):
            state = "EXECUTED"
        else:
            state = "FAILED"
        return {"run_id": self.run.name, "mode": mode, "state": state, "exit_code": EXIT_CODES[state],
                "detection_frames": sum(r["frames"] for r in detection_rows),
                "cuda_detection_frames": sum(r.get("cuda_frames", 0) for r in detection_rows),
                "successful_detection_jobs": len(detection_rows),
                "failed_stages": sum(r["state"] == "FAILED" for r in self.status),
                "blocked_stages": sum(r["state"] == "BLOCKED" for r in self.status)}

    def run_selected(self):
        self.environment_preflight()
        root = resolve_dataset(self.config["dataset_root"])
        mode = self.config["mode"]
        if mode in {"PREFLIGHT", "SMOKE", "BENCHMARK"}:
            self.preflight()
            if mode == "PREFLIGHT":
                self.audit_dataset(root)
                return
            if not any(self.readiness.values()):
                self.log("dataset", "zhanto_highlights", "audit", "SKIPPED",
                         "Preflight blocked all selected methods before video decoding.")
                return
            self.audit_dataset(root)
            self.core("detection", self.config["detectors"], self.detect)
            self.core("tracking", self.config["trackers"], self.track)
        else:
            self.audit_dataset(root)
            for method in self.config["advanced_methods"]:
                group = groups_for(method)
                try:
                    self.advanced(group, method)
                except Exception as error:
                    self.record_error(group, method, "advanced", error)


def validate_detector_devices(rows, devices, expected):
    if len({expected.get(device) for device in devices}) != len(devices):
        return False
    pids = set()
    for device in devices:
        identity = expected.get(device)
        assigned = [r for r in rows if r["device"] == device]
        if not identity or not assigned or any(
                r.get("gpu_uuid") != identity or r.get("cuda_frames", 0) != r["frames"] or r["frames"] <= 0
                or r.get("inference_batches", 0) <= 0 for r in assigned):
            return False
        pids.add(assigned[0]["pid"])
        if any(r["pid"] != assigned[0]["pid"] for r in assigned):
            return False
    return len(pids) == len(devices)


def launch(root, config):
    validate(config)
    root = Path(root)
    release = root / "export" / "scoutra_experiments_results.zip"
    pointer = root / "latest_run.json"
    if config["mode"] == "REPORT":
        selected = config["report_run"] or (json_read(pointer)["run_id"] if pointer.exists() else None)
        path = find_run(root, selected)
        for name in ("summary.md", "method_comparison.md"):
            file = path / "reports" / name
            if file.is_file():
                print(file.read_text(encoding="utf-8"))
        return export_zip(path, release)
    directory = new_run(root)
    lab = Laboratory(root, config, directory)
    try:
        lab.run_selected()
    except Exception as error:
        lab.record_error("run", "orchestrator", "execution", error)
    finally:
        lab.reports()
        outcome = json_read(directory / "RUN_RESULT.json")
        release = export_zip(directory, release)
        json_write(pointer, {**outcome, "path": str(directory), "zip": str(release)})
    print((directory / "reports" / "summary.md").read_text(encoding="utf-8"))
    print(f"Run folder: {directory}\nZIP: {release}", flush=True)
    print(f"RUN_RESULT={outcome['state']} | exit={outcome['exit_code']} | "
          f"detector frames={outcome['detection_frames']} | CUDA frames={outcome['cuda_detection_frames']}", flush=True)
    if outcome["exit_code"]:
        print("See reports/failures.md in the saved ZIP.", flush=True)
    return release


if __name__ == "__main__":
    if len(sys.argv) != 4:
        raise SystemExit("Usage: lab.py OUTPUT_ROOT CONFIG_JSON EMBEDDED_SOURCE_SHA256")
    config = json_read(sys.argv[2])
    config["embedded_source_sha256"] = sys.argv[3]
    root = Path(sys.argv[1])
    launch(root, config)
    if config["mode"] != "REPORT":
        raise SystemExit(json_read(root / "latest_run.json")["exit_code"])
'''


In [ ]:
# Materialize exactly the embedded files, then start the selected run.
import hashlib
import json
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working")
if not WORK.is_dir():
    raise RuntimeError("Run this notebook in Kaggle. Codex Cloud CPU tests use the source modules separately.")

SOURCE_ID = hashlib.sha256(
    json.dumps(SOURCES, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()
runtime = WORK / ".scoutra_notebook" / SOURCE_ID
runtime.mkdir(parents=True, exist_ok=True)
for filename, code in SOURCES.items():
    compile(code, filename, "exec")
    target = runtime / filename
    target.write_text(code, encoding="utf-8")
results = WORK / "scoutra_experiments"
launch_config = runtime / "launch.json"
launch_config.write_text(json.dumps(CONFIG, indent=2) + "\n", encoding="utf-8")
from IPython.display import FileLink, Markdown, display

latest = results / "latest_run.json"
previous = latest.read_bytes() if latest.is_file() else None
completed = subprocess.run(
    [sys.executable, "-u", str(runtime / "lab.py"), str(results), str(launch_config), SOURCE_ID],
    cwd=runtime,
)
if latest.is_file() and latest.read_bytes() != previous:
    result = json.loads(latest.read_text())
    link = Path(result["zip"])
    display(FileLink(str(link.relative_to(Path.cwd())) if link.is_relative_to(Path.cwd()) else str(link)))
    display(Markdown((Path(result["path"]) / "reports" / "summary.md").read_text()))
elif RUN_MODE == "REPORT" and completed.returncode == 0:
    link = results / "export" / "scoutra_experiments_results.zip"
    display(FileLink(str(link.relative_to(Path.cwd())) if link.is_relative_to(Path.cwd()) else str(link)))
if completed.returncode:
    state = {1: "FAILED", 2: "BLOCKED"}.get(completed.returncode, "FAILED")
    raise RuntimeError(f"Scoutra run {state} (exit {completed.returncode}). See the run log and the ZIP above if produced.")

